# Assignment: Capacitated Facility Location Problem (CFLP)

## Comparison of Exact vs Tabu Search and Genetic Algorithm

**Course:** CS616 - Optimization Algorithms  
**Instructors:** Dr. Mahdi Khemakhem & Dr. Essra Aldessouki

---

## 1. Problem Description:

A company must decide which facilities to open to serve a set of clients at minimum total cost. Each facility has a **fixed opening cost** and a **limited capacity**. Each client has a **demand** that must be satisfied. Each client must be **assigned to exactly one open facility**, and the **total demand assigned to any facility cannot exceed its capacity**.

The total cost includes both **facility opening costs** and **client-to-facility assignment costs**.

The **Capacitated Facility Location Problem (CFLP)** is more challenging than the uncapacitated version because it adds capacity constraints, making it significantly harder for exact solvers while creating more realistic scenarios.

The CFLP appears in many real-world applications, such as:
- Warehouse location with storage limits
- Data center placement with processing capacity
- Service center allocation with staff limits
- Distribution hub planning with throughput constraints
- Healthcare facility location with patient capacity
- Retail store placement with inventory limits
- Supply chain network design with capacity restrictions

The global decision of a CFLP is to determine which facilities to open and how to assign clients to these facilities to minimize the total cost while respecting capacity constraints.

The CFLP is **strongly NP-hard**, making it computationally more challenging than the uncapacitated version. Exact methods like Mixed-Integer Programming (MIP) struggle with even moderate-sized instances, while metaheuristics such as **Tabu Search (TS)** and **Genetic Algorithms (GA)** can provide good solutions in reasonable time.

This assignment compares three approaches:
1. **Gurobi (Exact MIP)**: Guaranteed optimal solutions, but computationally expensive for large instances.
2. **Tabu Search (TS)**: A trajectory-based metaheuristic that uses a tabu list to avoid revisiting recent solutions and escape local optima.
3. **Genetic Algorithm (GA)**: A population-based metaheuristic inspired by natural evolution that explores the solution space through selection, crossover, and mutation.

The CFLP can be formulated as follows:

### Mathematical Formulation

To model the Capacitated Facility Location Problem, we define the following components:

#### Sets:
- $𝐼 = \{1, 2, \ldots, n\}$: Set of $n$ clients.
- $𝐽 = \{1, 2, \ldots, m\}$: Set of potential $m$ facilities.

#### Parameters:
- $f_j$: Fixed cost of opening facility $j \in J$.
- $c_{ij}$: Cost of assigning client $i \in I$ to facility $j \in J$.
- $q_j$: **Capacity** of facility $j \in J$ (maximum demand it can serve).
- $d_i$: **Demand** of client $i \in I$ (must be satisfied).

#### Decision Variables:
$$\forall i \in I, \forall j \in J, x_{𝑖𝑗} =
\begin{cases}
1 & \text{ if client } 𝑖 \text{ is assigned to facility } 𝑗 \\
0 & \text{ otherwise}
\end{cases}
$$
$$\forall j \in J, 𝑦_{𝑗} = \begin{cases}
1 & \text{ if facility } 𝑗 \text{ is opened} \\
0 & \text{ otherwise}
\end{cases}$$

#### Objective Function:

Minimize the total cost:

$$\text{Minimize } Z = \sum_{j \in J} f_j y_j + \sum_{i \in I} \sum_{j \in J} c_{ij} x_{ij}$$

#### Constraints:
1. **Demand satisfaction**: Each client is assigned to exactly one facility:
    $$\sum_{j \in J} x_{ij} = 1, \quad \forall i \in I$$

2. **Linking constraint**: Clients can only be assigned to open facilities:
   $$x_{ij} \leq y_{j}, \quad \forall i \in I, \forall j \in J$$

3. **Capacity constraint**: Total demand assigned to a facility cannot exceed its capacity:
   $$\sum_{i \in I} d_i \cdot x_{ij} \leq q_j \cdot y_j, \quad \forall j \in J$$

4. **Binary constraints** on decision variables:
   $$x_{ij} \in \{0, 1\}, \quad \forall i \in I, \forall j \in J$$
   $$y_{j} \in \{0, 1\}, \quad \forall j \in J$$

#### Problem Complexity:
The capacity constraints make CFLP significantly harder than uncapacitated FLP because:
- **Tighter LP relaxation**: Capacity constraints create stronger linear programming bounds
- **Combinatorial explosion**: Limited capacity forces more facilities to open
- **Packing problem**: Assigning clients becomes a bin-packing subproblem
- **Infeasibility risk**: Tight capacities can lead to infeasible partial solutions

This makes CFLP an excellent benchmark for comparing exact and metaheuristic methods!


---

## 2. Import Libraries

In this section we import the necessary libraries for data handling, optimization, and visualization.

If a library is not installed, the script will attempt to install it using `pip install`.

In [ ]:
# Try to import necessary libraries, install if not available
try:
    import numpy as np
except ImportError:
    import sys
    !{sys.executable} -m pip install numpy
    import numpy as np
NUMPY_AVAILABLE = True

try:
    import pandas as pd
except ImportError:
    import sys
    !{sys.executable} -m pip install pandas
    import pandas as pd
PANDAS_AVAILABLE = True

try:
    import matplotlib.pyplot as plt
except ImportError:
    import sys
    !{sys.executable} -m pip install matplotlib
    import matplotlib.pyplot as plt
MATPLOTLIB_AVAILABLE = True

try:
    import seaborn as sns
except ImportError:
    import sys
    !{sys.executable} -m pip install seaborn
    import seaborn as sns
SEABORN_AVAILABLE = True

try:
    from scipy.interpolate import make_interp_spline
except ImportError:
    import sys
    !{sys.executable} -m pip install scipy
    from scipy.interpolate import make_interp_spline
SCIPY_AVAILABLE = True

try:
    import gurobipy as gp
    from gurobipy import GRB
except ImportError:
    import sys
    !{sys.executable} -m pip install gurobipy
    import gurobipy as gp
    from gurobipy import GRB
GUROBI_AVAILABLE = True

import time 
import os
import warnings
import shutil 
import json
from typing import Tuple, List

warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

if NUMPY_AVAILABLE and PANDAS_AVAILABLE and MATPLOTLIB_AVAILABLE and SEABORN_AVAILABLE and SCIPY_AVAILABLE and GUROBI_AVAILABLE:
    print("✅ All libraries imported successfully.")
else:
    print("❌ Some libraries failed to import.")


---

## 3. Instance Generator and Loader

In this section, we generate challenging **Capacitated Facility Location Problem (CFLP)** instances that create meaningful computational challenges for both exact and metaheuristic algorithms. Our instance generator creates **144 test cases** across 3 size categories and 3 difficulty levels, following a validated economic model with realistic capacity constraints.

### 3.1 Instance Generation Methodology

The generator implements a **research-based ratio model** for CFLP following operations research best practices:

$$\sum_{j} f_j = \alpha \times \sum_{i} \min_{j} c_{ij}$$

where:
- $\sum_{j} f_j$ = total fixed costs across all facilities
- $\alpha$ = fixed-to-**optimal**-service cost **ratio** (NOT multiplier!)
- $\sum_{i} \min_{j} c_{ij}$ = **optimal service cost** (minimum cost per client, NOT sum of all pairs!)
- Individual facility costs: $f_j = \frac{\alpha \cdot \text{optimal\_service}}{n} \cdot \frac{\bar{c}_j}{\bar{c}_{global}} \cdot \eta_j$

**CRITICAL**: α is computed against the **optimal service cost** (what you'd pay if each client uses their nearest facility), NOT the sum of all service cost pairs. This ensures proper difficulty calibration!

**Key Insight**: Computational hardness comes from **balanced trade-offs**, not high absolute costs!
- When $\alpha \approx 0.5$: Fixed costs = 50% of optimal service → Open many facilities → **EASY**
- When $\alpha \approx 1.0$: Fixed costs = 100% of optimal service → True trade-offs → **HARDEST!**
- When $\alpha \approx 3.0$: Fixed costs = 300% of optimal service → Open few facilities → **MEDIUM**

This creates a **U-shaped difficulty curve** where moderate instances (α ≈ 1.0) are the most challenging.

### 3.2 Size Categories (Optimized for Free Gurobi + CFLP Hardness)

Instances are carefully sized to **stay within free Gurobi limits (2000 variables, 2000 constraints)** while maximizing computational difficulty through **capacity constraints**:

**Variable Count**: $n \times m + m$ (assignment variables + facility variables)
**Constraint Count**: $n + n \times m + m$ (demand + linking + capacity constraints)

| Category | Facilities | Clients | Variables | Constraints | Characteristics |
|----------|------------|---------|-----------|-------------|-----------------|
| **Small** | 15-20 | 50-80 | 750-1,620 | 815-1,700 | Solvable but challenging with capacity |
| **Medium** | 18-24 | 60-75 | 1,080-1,824 | 1,158-1,923 | Very challenging with tight capacities |
| **Large** | 20-28 | 65-70 | 1,320-1,988 | 1,405-2,086 | **Near limit**, capacity creates extreme difficulty |

### 3.3 Difficulty Levels (Capacity Tightness + Cost Ratio Model)

| Level | α Range | β (Capacity/Demand) | Algorithmic Challenge |
|-------|---------|---------------------|----------------------|
| **Easy** | [0.3, 0.6] | [1.4, 1.7] | Loose capacity + cheap fixed → Simple |
| **Moderate** | [0.8, 1.5] | [1.1, 1.3] | **HARDEST! Tight capacity + balanced costs** |
| **Expensive** | [2.0, 4.0] | [1.2, 1.5] | Expensive facilities + moderate capacity |

### 3.4 Expected Computational Behavior

**Gurobi (Exact Method)**:
- Small easy: 2-10 seconds
- Small moderate: 10-60 seconds (tight capacity creates extreme difficulty!)
- Medium moderate: 40-180 seconds
- Large moderate: 120-300+ seconds

**Tabu Search (TS)**:
- Small: 0.5-3 seconds
- Medium: 1-5 seconds
- Large: 2-8 seconds
- Gap pattern: Easy 2-6%, Moderate 8-18%, Expensive 4-10%

**Genetic Algorithm (GA)**:
- Small: 1-5 seconds
- Medium: 2-8 seconds
- Large: 3-12 seconds
- Gap pattern: Easy 3-8%, Moderate 10-20%, Expensive 5-12%

**Key Insight**: **Capacity tightness** (β ≈ 1.2) combined with **cost balance** (α ≈ 1.0) creates the HARDEST instances. Both TS and GA trade solution quality for speed compared to Gurobi.

### 3.5 Instance Distribution

The generator creates **144 total instances**:
- 48 instances per size category (small, medium, large)
- Within each size: 16 instances per difficulty level
- Ensures balanced evaluation across all 9 combinations (3×3)

### 3.6 Output Format

Each instance is saved as a **JSON file** with keys: `num_facilities`, `num_clients`, `fixed_costs`, `service_costs`, `facility_capacities`, `client_demands`, `client_coordinates`, `facility_coordinates`.


In [ ]:
def generate_fixed_costs_from_cost_matrix(service_costs: np.ndarray, alpha: float, 
                                         noise_range: tuple = (0.8, 1.2)) -> np.ndarray:
    """
    Generate fixed costs using α as RATIO of OPTIMAL service cost to fixed cost.
    
    CRITICAL: α is computed against OPTIMAL service cost (minimum per client),
    NOT against the sum of all service cost pairs!
    
    Research-based model: Total fixed costs should be α × optimal service costs
    This creates meaningful trade-offs:
        - α < 0.5: Fixed costs cheap → open many facilities → EASY
        - α ≈ 1.0: Fixed costs ≈ optimal service costs → hard trade-offs → HARDEST!
        - α > 2.0: Fixed costs expensive → open few facilities → MEDIUM
    
    Formula: 
        optimal_service = Σ(min_j c_ij)  ← Cost if each client uses nearest facility
        target_fixed = α × optimal_service
        f_j = (target_fixed / n) × (c̄_j / c̄_global) × η_j
    
    where:
        - α is the fixed-to-optimal-service cost ratio
        - optimal_service is the minimum achievable service cost
        - n is number of facilities (distributes total budget)
        - (c̄_j / c̄_global) scales by relative facility importance
        - η_j adds heterogeneity noise
    
    Parameters:
    -----------
    service_costs : np.ndarray
        Service cost matrix of shape (num_clients, num_facilities)
    alpha : float
        Fixed-to-optimal-service cost RATIO (0.3-0.6: easy, 0.8-1.5: hard, 2.0-4.0: medium)
    noise_range : tuple
        Range for multiplicative noise (default: ±20%)
    
    Returns:
    --------
    fixed_costs : np.ndarray
        Fixed costs for each facility
    """
    num_clients, num_facilities = service_costs.shape
    
    # CRITICAL FIX: Use OPTIMAL service cost (minimum per client) as baseline!
    # Not the sum of all pairs which is irrelevant to the actual problem
    optimal_service_cost = service_costs.min(axis=1).sum()
    
    # Target: Total fixed costs = α × OPTIMAL service cost
    # This is the CORRECT way to calibrate difficulty!
    target_total_fixed_cost = alpha * optimal_service_cost
    
    # Compute average cost per facility (for proportional distribution)
    avg_cost_per_facility = service_costs.mean(axis=0)
    global_avg_cost = avg_cost_per_facility.mean()
    
    # Distribute total budget proportionally to facility importance
    # Facilities serving expensive regions get higher fixed costs
    importance_weights = avg_cost_per_facility / global_avg_cost
    base_fixed_cost = target_total_fixed_cost / num_facilities
    
    # Scale by importance and add heterogeneity
    noise_factors = np.random.uniform(noise_range[0], noise_range[1], size=num_facilities)
    fixed_costs = base_fixed_cost * importance_weights * noise_factors
    
    # Normalize to maintain target total (compensate for noise)
    fixed_costs = fixed_costs * (target_total_fixed_cost / fixed_costs.sum())
    
    # Ensure positive and reasonable values
    fixed_costs = np.maximum(fixed_costs, 1.0)
    
    return fixed_costs


def generate_instances(folder: str = "cflp_instances", num_total_instances: int = 144, seed: int = 42):
    """
    Generate Capacitated Facility Location Problem (CFLP) instances with scientific rigor.
    
    Creates instances across 3 size categories (small, medium, large) and 
    3 difficulty levels (easy, moderate, expensive), following the methodology:
    
    Size Categories:
    - Small: 2-5 facilities, 30-60 clients
    - Medium: 6-10 facilities, 80-150 clients  
    - Large: 12-20 facilities, 200-400 clients
    
    Difficulty Levels (via α parameter):
    - Easy: α ∈ [5, 10] - Low fixed costs, most facilities profitable
    - Moderate: α ∈ [10, 20] - Balanced trade-offs
    - Expensive: α ∈ [20, 40] - High fixed costs, selective openings
    
    Parameters:
    -----------
    folder : str
        Directory to save generated instances
    num_total_instances : int
        Total number of instances to generate (distributed evenly)
    seed : int
        Random seed for reproducibility
    
    Returns:
    --------
    None (instances saved to disk with summary Excel file)
    """
    np.random.seed(seed)
    
    # ==================================================================================
    # CLEANUP: Remove all old instances before generating new ones
    # ==================================================================================
    print(f"\n{'='*80}")
    print(f"CLEANING UP OLD INSTANCES")
    print(f"{'='*80}")
    
    if os.path.exists(folder):
        file_count = 0
        dir_count = 0
        
        # Count and delete all contents
        for filename in os.listdir(folder):
            file_path = os.path.join(folder, filename)
            try:
                if os.path.isfile(file_path):
                    os.unlink(file_path)
                    file_count += 1
                elif os.path.isdir(file_path):
                    shutil.rmtree(file_path)
                    dir_count += 1
            except Exception as e:
                print(f"⚠️  Warning: Could not delete {file_path}: {e}")
        
        # Remove and recreate folder to ensure complete cleanup
        try:
            shutil.rmtree(folder)
            print(f"✓ Removed existing folder '{folder}'")
            print(f"  - Deleted {file_count} file(s)")
            print(f"  - Deleted {dir_count} subdirectory(ies)")
        except Exception as e:
            print(f"⚠️  Warning: Could not remove folder {folder}: {e}")
    else:
        print(f"✓ Folder '{folder}' does not exist (first run)")
    
    # Create fresh output folder
    os.makedirs(folder, exist_ok=True)
    print(f"✓ Created fresh folder '{folder}'\n")

    # ==================================================================================
    # CFLP DIFFICULTY MODEL (Capacity Tightness + Cost Ratio)
    # ==================================================================================
    # Key factors creating computational hardness in CFLP:
    # 1. CAPACITY TIGHTNESS (β): Ratio of total capacity to total demand
    #    - β < 1.2: Very tight → forces many facilities → EXTREMELY HARD
    #    - β ≈ 1.3: Moderate → interesting trade-offs → HARD
    #    - β > 1.5: Loose → capacity less binding → easier
    # 2. COST RATIO (α): Fixed costs relative to service costs
    # 3. PROBLEM SIZE: Limited by free Gurobi (2000 vars, 2000 constraints)
    #
    # Variables: n*m + m  |  Constraints: n + n*m + m
    # Must keep: (n*m + m) ≤ 2000  AND  (n + n*m + m) ≤ 2000
    size_settings = {
        "small": {
            "fac_range": (15, 21),      # 15-20 facilities
            "cli_range": (50, 81),      # 50-80 clients → Max ~1,620 vars, ~1,700 constraints
            "alpha_ranges": {
                "easy": (0.3, 0.6),      # Cheap facilities + loose capacity
                "moderate": (0.8, 1.5),  # Balanced costs + tight capacity → HARDEST!
                "expensive": (2.0, 4.0)  # Expensive facilities + moderate capacity
            },
            "beta_ranges": {
                "easy": (1.4, 1.7),      # Loose capacity (40-70% slack)
                "moderate": (1.1, 1.3),  # Tight capacity (10-30% slack) → HARDEST!
                "expensive": (1.2, 1.5)  # Moderate capacity
            }
        },
        "medium": {
            "fac_range": (18, 25),      # 18-24 facilities
            "cli_range": (60, 76),      # 60-75 clients → Max ~1,824 vars, ~1,923 constraints
            "alpha_ranges": {
                "easy": (0.3, 0.6),
                "moderate": (0.8, 1.5),  # HARDEST with tight capacity!
                "expensive": (2.0, 4.0)
            },
            "beta_ranges": {
                "easy": (1.4, 1.7),
                "moderate": (1.1, 1.3),  # Very challenging!
                "expensive": (1.2, 1.5)
            }
        },
        "large": {
            "fac_range": (20, 28),      # 20-27 facilities (reduced max to stay within limits)
            "cli_range": (65, 69),      # 65-68 clients → Max ~1,863 vars, ~1,944 constraints (safe margin!)
            "alpha_ranges": {
                "easy": (0.3, 0.6),
                "moderate": (0.8, 1.5),  # HARDEST! Near Gurobi free limit + tight capacity
                "expensive": (2.0, 4.0)
            },
            "beta_ranges": {
                "easy": (1.4, 1.7),
                "moderate": (1.1, 1.3),  # Extremely challenging!
                "expensive": (1.2, 1.5)
            }
        }
    }

    # Calculate instances per category (evenly distributed)
    num_sizes = len(size_settings)
    num_difficulties = 3
    instances_per_combination = num_total_instances // (num_sizes * num_difficulties)
    
    instances_info = []
    instance_id = 1

    print(f"Generating {num_total_instances} CFLP instances...")
    print(f"Distribution: {instances_per_combination} per size-difficulty combination")
    print(f"All instances comply with free Gurobi limits (≤2000 vars, ≤2000 constraints)\n")
    
    # Generate instances for each size-difficulty combination
    for size, params in size_settings.items():
        for difficulty, alpha_range in params["alpha_ranges"].items():
            beta_range = params["beta_ranges"][difficulty]
            
            for rep in range(instances_per_combination):
                # Randomize problem dimensions
                num_facilities = np.random.randint(*params["fac_range"])
                num_clients = np.random.randint(*params["cli_range"])
                
                # Verify free Gurobi compliance
                num_vars = num_clients * num_facilities + num_facilities
                num_constraints = num_clients + num_clients * num_facilities + num_facilities
                
                if num_vars > 2000 or num_constraints > 2000:
                    print(f"⚠️  WARNING: Instance too large! Vars={num_vars}, Constraints={num_constraints}")
                    continue
                
                # Sample α (cost ratio) and β (capacity tightness) from difficulty range
                alpha = np.random.uniform(*alpha_range)
                beta = np.random.uniform(*beta_range)

                # Generate UNIQUE random 2D coordinates using grid-based approach
                # This ensures no overlapping points and creates diverse spatial patterns
                grid_size = 1000  # Large grid for unique positions
                
                # Generate unique client positions
                clients = []
                used_positions = set()
                attempts = 0
                while len(clients) < num_clients and attempts < num_clients * 10:
                    x = np.random.uniform(0, grid_size)
                    y = np.random.uniform(0, grid_size)
                    # Round to ensure uniqueness but maintain diversity
                    pos_key = (round(x, 1), round(y, 1))
                    if pos_key not in used_positions:
                        clients.append([x, y])
                        used_positions.add(pos_key)
                    attempts += 1
                clients = np.array(clients)
                
                # Generate unique facility positions (separate from clients)
                facilities = []
                attempts = 0
                while len(facilities) < num_facilities and attempts < num_facilities * 10:
                    x = np.random.uniform(0, grid_size)
                    y = np.random.uniform(0, grid_size)
                    pos_key = (round(x, 1), round(y, 1))
                    if pos_key not in used_positions:
                        facilities.append([x, y])
                        used_positions.add(pos_key)
                    attempts += 1
                facilities = np.array(facilities)

                # Compute service costs: distance + perturbation
                # Perturbation creates problem structure, not difficulty directly
                service_costs = np.zeros((num_clients, num_facilities))
                
                # Use CONSTANT moderate noise across all difficulties
                # Difficulty comes from α (cost ratio) and problem size, NOT noise!
                base_noise = 0.15  # 15% noise is realistic for real-world problems
                
                for i in range(num_clients):
                    for j in range(num_facilities):
                        euclidean_dist = np.linalg.norm(clients[i] - facilities[j])
                        # Add proportional noise (15% of distance)
                        perturbation = euclidean_dist * np.random.uniform(0, base_noise)
                        service_costs[i, j] = euclidean_dist + perturbation

                # ===================================================================
                # GENERATE DEMANDS AND CAPACITIES FOR CFLP
                # ===================================================================
                # Client demands: uniform random in [5, 20] (realistic range)
                client_demands = np.random.uniform(5, 20, size=num_clients)
                total_demand = client_demands.sum()
                
                # Facility capacities based on β (capacity tightness ratio)
                # β = total_capacity / total_demand
                # Lower β → tighter capacity → harder problem!
                target_total_capacity = beta * total_demand
                
                # Distribute capacity across facilities with heterogeneity
                # Base capacity per facility
                base_capacity = target_total_capacity / num_facilities
                
                # Add heterogeneity: some facilities larger, some smaller
                capacity_factors = np.random.uniform(0.7, 1.3, size=num_facilities)
                facility_capacities = base_capacity * capacity_factors
                
                # Normalize to maintain exact β ratio
                facility_capacities = facility_capacities * (target_total_capacity / facility_capacities.sum())
                
                # Ensure all capacities can serve at least 2-3 clients
                min_capacity = 2 * client_demands.mean()
                facility_capacities = np.maximum(facility_capacities, min_capacity)
                
                # Recalculate actual β after adjustments
                actual_beta = facility_capacities.sum() / total_demand
                
                # ===================================================================
                # GENERATE FIXED COSTS USING α RATIO MODEL
                # ===================================================================
                fixed_costs = generate_fixed_costs_from_cost_matrix(
                    service_costs, 
                    alpha=alpha, 
                    noise_range=(0.8, 1.2)  # ±20% heterogeneity in fixed costs
                )

                # Create instance filename
                instance_name = f"{instance_id:03d}_cflp_{size}_{difficulty}_fac{num_facilities}_cli{num_clients}.json"
                instance_path = os.path.join(folder, instance_name)

                # Prepare instance data with capacities and demands
                instance_data = {
                    "num_facilities": int(num_facilities),
                    "num_clients": int(num_clients),
                    "size": size,
                    "difficulty": difficulty,
                    "alpha": round(alpha, 3),
                    "beta": round(actual_beta, 3),
                    "total_demand": round(total_demand, 2),
                    "total_capacity": round(facility_capacities.sum(), 2),
                    "fixed_costs": fixed_costs.tolist(),
                    "service_costs": service_costs.tolist(),
                    "facility_capacities": facility_capacities.tolist(),
                    "client_demands": client_demands.tolist(),
                    "client_coordinates": clients.tolist(),
                    "facility_coordinates": facilities.tolist(),
                    "num_variables": int(num_vars),
                    "num_constraints": int(num_constraints)
                }

                # Save instance to JSON
                with open(instance_path, "w") as f:
                    json.dump(instance_data, f, indent=2)

                # Store summary information
                instances_info.append({
                    "instance_name": instance_name,
                    "instance_id": instance_id,
                    "size": size,
                    "difficulty": difficulty,
                    "num_facilities": num_facilities,
                    "num_clients": num_clients,
                    "num_variables": num_vars,
                    "num_constraints": num_constraints,
                    "alpha": round(alpha, 3),
                    "beta": round(actual_beta, 3),
                    "total_demand": round(total_demand, 2),
                    "total_capacity": round(facility_capacities.sum(), 2),
                    "capacity_slack_pct": round((actual_beta - 1.0) * 100, 1),
                    "avg_fixed_cost": round(fixed_costs.mean(), 2),
                    "avg_service_cost": round(service_costs.mean(), 2),
                    "avg_demand": round(client_demands.mean(), 2),
                    "avg_capacity": round(facility_capacities.mean(), 2)
                })
                
                # Create visualization with correct coordinate bounds
                plt.figure(figsize=(10, 10))
                
                # Adjust marker sizes based on instance size for visibility
                if num_clients > 500:
                    client_size = 5
                    facility_size = 50
                elif num_clients > 200:
                    client_size = 10
                    facility_size = 80
                else:
                    client_size = 20
                    facility_size = 120
                
                plt.scatter(facilities[:, 0], facilities[:, 1], 
                           c='red', marker='s', s=facility_size, alpha=0.8, 
                           edgecolors='darkred', linewidths=2, 
                           label=f'Facilities (n={num_facilities})')
                plt.scatter(clients[:, 0], clients[:, 1], 
                           c='blue', marker='o', s=client_size, alpha=0.5, 
                           label=f'Clients (n={num_clients})')
                
                plt.title(f"CFLP Instance {instance_id:03d}: {size.capitalize()}-{difficulty.capitalize()}\n"
                         f"Facilities: {num_facilities}, Clients: {num_clients}, α={alpha:.2f}, β={actual_beta:.2f}",
                         fontsize=14, fontweight='bold')
                plt.xlabel("X Coordinate", fontsize=12)
                plt.ylabel("Y Coordinate", fontsize=12)
                
                # Set proper limits based on actual coordinate space (0-1000)
                plt.xlim(-50, 1050)
                plt.ylim(-50, 1050)
                
                plt.legend(loc='upper right', fontsize=10)
                plt.grid(alpha=0.3, linestyle='--')
                plt.tight_layout()
                
                # Save visualization
                viz_path = os.path.join(folder, instance_name.replace(".json", ".png"))
                plt.savefig(viz_path, dpi=150, bbox_inches='tight')
                plt.close()

                instance_id += 1

    # Create summary Excel file
    summary_df = pd.DataFrame(instances_info)
    summary_path = os.path.join(folder, "instances_summary.xlsx")
    
    with pd.ExcelWriter(summary_path, engine='openpyxl') as writer:
        # Full summary
        summary_df.to_excel(writer, sheet_name='All Instances', index=False)
        
        # Statistics by size
        size_stats = summary_df.groupby('size').agg({
            'num_facilities': ['min', 'max', 'mean'],
            'num_clients': ['min', 'max', 'mean'],
            'alpha': ['min', 'max', 'mean'],
            'beta': ['min', 'max', 'mean'],
            'num_variables': ['min', 'max', 'mean'],
            'num_constraints': ['min', 'max', 'mean'],
            'instance_id': 'count'
        }).round(2)
        size_stats.to_excel(writer, sheet_name='Size Statistics')
        
        # Statistics by difficulty
        diff_stats = summary_df.groupby('difficulty').agg({
            'alpha': ['min', 'max', 'mean'],
            'beta': ['min', 'max', 'mean'],
            'total_demand': ['min', 'max', 'mean'],
            'total_capacity': ['min', 'max', 'mean'],
            'instance_id': 'count'
        }).round(2)
        diff_stats.to_excel(writer, sheet_name='Difficulty Statistics')
        
        # Size-Difficulty matrix
        matrix = summary_df.pivot_table(
            values='instance_id', 
            index='size', 
            columns='difficulty', 
            aggfunc='count',
            fill_value=0
        )
        matrix.to_excel(writer, sheet_name='Distribution Matrix')
    
    print(f"\nSuccessfully generated {len(instances_info)} instances in '{folder}/'")
    print(f"   Summary saved to: {summary_path}")
    print(f"\nDistribution Summary:")
    print(summary_df.groupby(['size', 'difficulty']).size().unstack(fill_value=0))


def load_cflp_instance(file_path: str) -> Tuple[int, int, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Load a Capacitated Facility Location Problem instance from a JSON file.
    
    Parameters:
    -----------
    file_path : str
        Path to the JSON file containing the instance data
        
    Returns:
    --------
    num_facilities : int
        Number of facilities
    num_clients : int
        Number of clients
    fixed_costs : np.ndarray
        Fixed costs for each facility (shape: num_facilities)
    service_costs : np.ndarray
        Service costs matrix (shape: num_clients × num_facilities)
    facility_capacities : np.ndarray
        Capacity of each facility (shape: num_facilities)
    client_demands : np.ndarray
        Demand of each client (shape: num_clients)
    """
    with open(file_path, 'r') as f:
        instance_data = json.load(f)
    
    num_facilities = instance_data['num_facilities']
    num_clients = instance_data['num_clients']
    fixed_costs = np.array(instance_data['fixed_costs'])
    service_costs = np.array(instance_data['service_costs'])
    facility_capacities = np.array(instance_data['facility_capacities'])
    client_demands = np.array(instance_data['client_demands'])
    
    return num_facilities, num_clients, fixed_costs, service_costs, facility_capacities, client_demands

print("✅ CFLP instance generation module loaded successfully.")

# Uncomment the line below to test the instance generation
# generate_instances(folder="cflp_instances", num_total_instances=144, seed=42)

In [ ]:
def generate_all_instances(output_folder: str = "cflp_instances",
                           sizes=None,
                           difficulties=None,
                           instances_per_combo: int = 16,
                           seed: int = 42):
    """
    Generate all CFLP instances across size and difficulty combinations.

    This is the entry point called by main(). It delegates to generate_instances()
    which creates instances across small/medium/large × easy/moderate/expensive.

    Parameters:
    -----------
    output_folder : str
        Folder where JSON instance files will be saved.
    sizes : list, optional
        List of size categories (default: ['small', 'medium', 'large']).
    difficulties : list, optional
        List of difficulty levels (default: ['easy', 'moderate', 'expensive']).
    instances_per_combo : int
        Number of instances per (size, difficulty) combination.
    seed : int
        Random seed for reproducibility.
    """
    if sizes is None:
        sizes = ['small', 'medium', 'large']
    if difficulties is None:
        difficulties = ['easy', 'moderate', 'expensive']

    num_total = len(sizes) * len(difficulties) * instances_per_combo
    generate_instances(folder=output_folder, num_total_instances=num_total, seed=seed)


print("✅ generate_all_instances() registered successfully.")


---

## 4. Solution Encoding (Representation) and Utility Functions

In this section, we define the solution encoding for the Facility Location Problem (FLP) and implement utility functions to evaluate and manipulate solutions. 

The solution is represented as a dictionary with two components:
1. `facilities`: A binary list indicating which facilities are open (1) or closed (0).
2. `assignments`: A list where each element represents the index of the facility assigned to each client.

For example, a dictionary representing a solution might look like this:
{
  "facilities": [1, 0, 1, 0, 1],
  "assignments": [0, 2, 4, 2, 0, 2, 4]
}
- Facilities: [1, 0, 1, 0, 1] (facilities 1, 3, and 5 are open)
- Assignments: [0, 2, 4, 2, 0, 2, 4] (client 1 assigned to facility 1, client 2 to facility 3, client 3 to facility 5, etc.)

The functions related to solution encoding include:
- `compute_total_cost`: Calculate total cost of a solution
- `is_feasible`: Check if a solution is feasible
- `assign_clients_to_nearest_open_facility`: Assign clients to the nearest open facility
- `copy_solution`: Create a deep copy of a solution
- `greedy_initial_solution`: Generate an initial solution using a greedy heuristic. It consists to:
    1. Start with all facilities closed.
    2. Iteratively open the facility that provides the largest cost reduction until no further improvement is possible.
    3. Assign clients to the nearest open facility.

In [ ]:
def compute_total_cost(fixed_costs: np.ndarray, service_costs: np.ndarray, 
                       facility_opened: np.ndarray, client_assignments: np.ndarray) -> float:
    """
    Compute the total cost of a given solution.
    
    Parameters:
    -----------
    fixed_costs : np.ndarray
        Fixed costs for each facility (shape: num_facilities)
    service_costs : np.ndarray
        Service costs matrix (shape: num_clients × num_facilities)
    facility_opened : np.ndarray
        Binary array indicating which facilities are opened (shape: num_facilities)
    client_assignments : np.ndarray
        Array indicating the assigned facility for each client (shape: num_clients)
        
    Returns:
    --------
    total_cost : float
        Total cost of the solution
    """
    total_fixed_cost = np.sum(fixed_costs * facility_opened)
    total_service_cost = np.sum(service_costs[np.arange(len(client_assignments)), client_assignments])
    
    total_cost = total_fixed_cost + total_service_cost
    return total_cost

def is_feasible(facility_opened: np.ndarray, client_assignments: np.ndarray,
                facility_capacities: np.ndarray, client_demands: np.ndarray) -> bool:
    """
    Check if a given solution is feasible for CFLP.
    
    Parameters:
    -----------
    facility_opened : np.ndarray
        Binary array indicating which facilities are opened (shape: num_facilities)
    client_assignments : np.ndarray
        Array indicating the assigned facility for each client (shape: num_clients)
    facility_capacities : np.ndarray
        Capacity of each facility (shape: num_facilities)
    client_demands : np.ndarray
        Demand of each client (shape: num_clients)
        
    Returns:
    --------
    feasible : bool
        True if the solution is feasible, False otherwise
    """
    # Check if clients are only assigned to open facilities
    for assigned_facility in client_assignments:
        if facility_opened[assigned_facility] == 0:
            return False
    
    # Check capacity constraints
    for j in range(len(facility_opened)):
        if facility_opened[j] == 1:
            assigned_clients = np.where(client_assignments == j)[0]
            total_demand = client_demands[assigned_clients].sum()
            if total_demand > facility_capacities[j] + 1e-6:  # Small tolerance for floating point
                return False
    
    return True

def assign_clients_to_nearest_open_facility(service_costs: np.ndarray, facility_opened: np.ndarray,
                                            facility_capacities: np.ndarray, client_demands: np.ndarray) -> np.ndarray:
    """
    Assign each client to the nearest opened facility respecting capacity constraints (CFLP).
    Uses a fast capacity-aware greedy heuristic with demand-based sorting.
    
    Parameters:
    -----------
    service_costs : np.ndarray
        Service costs matrix (shape: num_clients × num_facilities)
    facility_opened : np.ndarray
        Binary array indicating which facilities are opened (shape: num_facilities)
    facility_capacities : np.ndarray
        Capacity of each facility (shape: num_facilities)
    client_demands : np.ndarray
        Demand of each client (shape: num_clients)
        
    Returns:
    --------
    client_assignments : np.ndarray
        Array indicating the assigned facility for each client (shape: num_clients)
    """
    num_clients = service_costs.shape[0]
    opened_indices = np.where(facility_opened == 1)[0]
    
    if len(opened_indices) == 0:
        raise ValueError("No facilities are open!")
    
    # Initialize assignments and track remaining capacity
    client_assignments = np.full(num_clients, -1, dtype=int)
    remaining_capacity = facility_capacities.copy()
    
    # Get costs only for opened facilities
    costs_to_open = service_costs[:, opened_indices]
    
    # Assign each client to nearest facility with remaining capacity
    # Process clients in natural order (no sorting) for flexibility
    for client_idx in range(num_clients):
        costs = costs_to_open[client_idx]
        sorted_fac_positions = np.argsort(costs)
        
        assigned = False
        for pos in sorted_fac_positions:
            fac_idx = opened_indices[pos]
            if remaining_capacity[fac_idx] >= client_demands[client_idx]:
                client_assignments[client_idx] = fac_idx
                remaining_capacity[fac_idx] -= client_demands[client_idx]
                assigned = True
                break
        
        if not assigned:
            # Infeasible - assign to nearest anyway (will be caught by is_feasible)
            nearest_fac = opened_indices[sorted_fac_positions[0]]
            client_assignments[client_idx] = nearest_fac
    
    return client_assignments

def copy_solution(facility_opened: np.ndarray, client_assignments: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """
    Create a deep copy of a solution.
    
    Parameters:
    -----------
    facility_opened : np.ndarray
        Binary array indicating which facilities are opened (shape: num_facilities)
    client_assignments : np.ndarray
        Array indicating the assigned facility for each client (shape: num_clients)
        
    Returns:
    --------
    facility_opened_copy : np.ndarray
        Copy of the facility opened array
    client_assignments_copy : np.ndarray
        Copy of the client assignments array
    """
    facility_opened_copy = np.copy(facility_opened)
    client_assignments_copy = np.copy(client_assignments)
    
    return facility_opened_copy, client_assignments_copy

def greedy_initial_solution(num_clients: int, num_facilities: int, service_costs: np.ndarray,
                           facility_capacities: np.ndarray, client_demands: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """
    Generate an initial solution using a greedy heuristic for CFLP.
    
    Parameters:
    -----------
    num_clients : int
        Number of clients
    num_facilities : int
        Number of facilities
    service_costs : np.ndarray
        Service costs matrix (shape: num_clients × num_facilities)
    facility_capacities : np.ndarray
        Capacity of each facility (shape: num_facilities)
    client_demands : np.ndarray
        Demand of each client (shape: num_clients)

    Returns:
    --------
    facility_opened : np.ndarray
        Binary array indicating which facilities are opened (shape: num_facilities)
    client_assignments : np.ndarray
        Array indicating the assigned facility for each client (shape: num_clients)
    """
    # Simple greedy approach: open facilities with best average service cost
    # until we have enough capacity to serve all clients
    
    # Calculate average service cost for each facility
    avg_service_cost_per_facility = service_costs.mean(axis=0)
    
    # Sort facilities by average service cost (ascending - lower is better)
    sorted_by_cost = np.argsort(avg_service_cost_per_facility)
    
    # Start with all facilities closed
    facility_opened = np.zeros(num_facilities, dtype=int)
    total_demand = np.sum(client_demands)
    
    # Open facilities in order of lowest average service cost until feasible
    for fac_idx in sorted_by_cost:
        facility_opened[fac_idx] = 1
        
        # Try assigning clients to open facilities
        client_assignments = assign_clients_to_nearest_open_facility(
            service_costs, facility_opened, facility_capacities, client_demands
        )
        
        # Check if feasible
        if is_feasible(facility_opened, client_assignments, facility_capacities, client_demands):
            # We have a feasible solution - stop here!
            break
    
    # If still not feasible (rare case), open all facilities
    if not is_feasible(facility_opened, client_assignments, facility_capacities, client_demands):
        facility_opened = np.ones(num_facilities, dtype=int)
        client_assignments = assign_clients_to_nearest_open_facility(
            service_costs, facility_opened, facility_capacities, client_demands
        )
    
    return facility_opened, client_assignments

print("✅ CFLP utility functions module loaded successfully.")

# Example usage: Uncomment the the following lines to load an instance and test utility functions
# file_path = "cflp_instances/001_cflp_small_easy_fac15_cli50.json"
# num_facilities, num_clients, fixed_costs, service_costs, facility_capacities, client_demands = load_cflp_instance(file_path)
# # combine to a dictionary
# instance = {
#     "num_facilities": num_facilities,
#     "num_clients": num_clients,
#     "fixed_costs": fixed_costs,
#     "service_costs": service_costs,
#     "facility_capacities": facility_capacities,
#     "client_demands": client_demands
# }
# # Generate initial solution
# facility_opened, client_assignments = greedy_initial_solution(
#     num_clients=instance["num_clients"], 
#     num_facilities=instance["num_facilities"], 
#     service_costs=instance["service_costs"],
#     facility_capacities=instance["facility_capacities"],
#     client_demands=instance["client_demands"]
# )
# # Display solution summary (just the first and last ten assignments)
# print(f"Facility opened: {facility_opened}")
# print(f"Client assignments: {client_assignments[:10]} ... {client_assignments[-10:]}")

# # Test total cost computation
# total_cost = compute_total_cost(
#     fixed_costs=instance["fixed_costs"], 
#     service_costs=instance["service_costs"], 
#     facility_opened=facility_opened, 
#     client_assignments=client_assignments
# )
# print(f"Total cost of the initial solution: {total_cost}")

# # Test feasibility check
# feasible = is_feasible(
#     facility_opened=facility_opened, 
#     client_assignments=client_assignments,
#     facility_capacities=instance["facility_capacities"],
#     client_demands=instance["client_demands"]
# )
# print(f"Is the initial solution feasible? {'Yes' if feasible else 'No'}")


---

## 5. Tabu Search (TS) Implementation

In this section, we implement the **Tabu Search (TS)** algorithm to solve the Capacitated Facility Location Problem (CFLP). Tabu Search is a trajectory-based metaheuristic that enhances local search by using a **tabu list** — a memory structure that records recently visited moves to prevent cycling and force the search to explore new areas of the solution space.

The TS algorithm consists of the following key components:
1. **Move Structures**: Define different ways to modify the current solution (open/close facilities, swap facilities, reassign clients).
2. **Tabu List**: A short-term memory that forbids reversing recent moves for a specified number of iterations (the **tabu tenure**).
3. **Aspiration Criterion**: Allows a tabu move to be accepted if it leads to a solution better than the current best known solution.
4. **Candidate Evaluation**: At each iteration, all admissible moves (non-tabu or satisfying aspiration) are evaluated and the best is selected — even if it does not improve the current solution.
5. **Stopping Criterion**: Maximum iterations or no improvement for a defined number of consecutive iterations.


### 5.1 Move Generation Structures

We define four types of moves to explore the solution space. Each move modifies the current solution and is identified by a unique key used in the tabu list:

a) **Move Type 1 — Open Facility** (`generate_open_moves`):
- Select a closed facility and open it; reassign clients to the nearest feasible open facility.
- Move key: `('open', j)` where `j` is the index of the newly opened facility.
- Reversal: `('close', j)` — closing the same facility is tabu for `tabu_tenure` iterations.

b) **Move Type 2 — Close Facility** (`generate_close_moves`):
- Select an open facility and close it (requires ≥ 2 open facilities); reassign clients.
- Move key: `('close', j)` where `j` is the index of the closed facility.
- Reversal: `('open', j)` — reopening the same facility is tabu.

c) **Move Type 3 — Swap Facilities** (`generate_swap_moves`):
- Close one open facility and simultaneously open one closed facility; reassign clients.
- Move key: `('swap', j_open, j_close)` where `j_open` is closed and `j_close` is opened.
- Reversal: `('swap', j_close, j_open)` — reversing the swap is tabu.

d) **Move Type 4 — Reassign Client** (`generate_reassign_moves`):
- Move a client from its current facility to a different open facility (capacity check required).
- Move key: `('reassign', client, old_facility, new_facility)`.
- Reversal: `('reassign', client, new_facility, old_facility)` — moving the client back is tabu.

**Key principle**: The tabu list stores the *inverse* (reversal) of each applied move, preventing it from being undone for `tabu_tenure` iterations. The **aspiration criterion** overrides the tabu status if the move leads to a solution strictly better than the best known.


In [ ]:
def generate_open_moves(facility_opened: np.ndarray, service_costs: np.ndarray,
                        facility_capacities: np.ndarray, client_demands: np.ndarray,
                        max_moves: int = 20) -> List[Tuple]:
    """
    Generate move candidates by opening a closed facility (Tabu Search move type 1).

    Parameters:
    -----------
    facility_opened : np.ndarray
        Binary array indicating which facilities are opened (shape: num_facilities)
    service_costs : np.ndarray
        Service costs matrix (shape: num_clients × num_facilities)
    facility_capacities : np.ndarray
        Capacity of each facility (shape: num_facilities)
    client_demands : np.ndarray
        Demand of each client (shape: num_clients)
    max_moves : int
        Maximum number of candidate moves to generate

    Returns:
    --------
    candidates : List[Tuple]
        List of (new_facility_opened, new_client_assignments, move_key) tuples
    """
    candidates = []
    closed_indices = np.where(facility_opened == 0)[0]
    np.random.shuffle(closed_indices)

    for j in closed_indices:
        if len(candidates) >= max_moves:
            break
        new_opened = facility_opened.copy()
        new_opened[j] = 1
        new_assignments = assign_clients_to_nearest_open_facility(
            service_costs, new_opened, facility_capacities, client_demands
        )
        if is_feasible(new_opened, new_assignments, facility_capacities, client_demands):
            move_key = ('open', int(j))
            candidates.append((new_opened, new_assignments, move_key))

    return candidates


def generate_close_moves(facility_opened: np.ndarray, service_costs: np.ndarray,
                         facility_capacities: np.ndarray, client_demands: np.ndarray,
                         max_moves: int = 20) -> List[Tuple]:
    """
    Generate move candidates by closing an open facility (Tabu Search move type 2).

    Parameters:
    -----------
    facility_opened : np.ndarray
        Binary array indicating which facilities are opened (shape: num_facilities)
    service_costs : np.ndarray
        Service costs matrix (shape: num_clients × num_facilities)
    facility_capacities : np.ndarray
        Capacity of each facility (shape: num_facilities)
    client_demands : np.ndarray
        Demand of each client (shape: num_clients)
    max_moves : int
        Maximum number of candidate moves to generate

    Returns:
    --------
    candidates : List[Tuple]
        List of (new_facility_opened, new_client_assignments, move_key) tuples
    """
    candidates = []
    opened_indices = np.where(facility_opened == 1)[0]

    # Must keep at least one facility open
    if len(opened_indices) <= 1:
        return candidates

    np.random.shuffle(opened_indices)
    for j in opened_indices:
        if len(candidates) >= max_moves:
            break
        new_opened = facility_opened.copy()
        new_opened[j] = 0
        new_assignments = assign_clients_to_nearest_open_facility(
            service_costs, new_opened, facility_capacities, client_demands
        )
        if is_feasible(new_opened, new_assignments, facility_capacities, client_demands):
            move_key = ('close', int(j))
            candidates.append((new_opened, new_assignments, move_key))

    return candidates


def generate_swap_moves(facility_opened: np.ndarray, service_costs: np.ndarray,
                        facility_capacities: np.ndarray, client_demands: np.ndarray,
                        max_moves: int = 20) -> List[Tuple]:
    """
    Generate move candidates by swapping an open and a closed facility (Tabu Search move type 3).

    Parameters:
    -----------
    facility_opened : np.ndarray
        Binary array indicating which facilities are opened (shape: num_facilities)
    service_costs : np.ndarray
        Service costs matrix (shape: num_clients × num_facilities)
    facility_capacities : np.ndarray
        Capacity of each facility (shape: num_facilities)
    client_demands : np.ndarray
        Demand of each client (shape: num_clients)
    max_moves : int
        Maximum number of candidate moves to generate

    Returns:
    --------
    candidates : List[Tuple]
        List of (new_facility_opened, new_client_assignments, move_key) tuples
    """
    candidates = []
    opened_indices = np.where(facility_opened == 1)[0]
    closed_indices = np.where(facility_opened == 0)[0]

    if len(opened_indices) == 0 or len(closed_indices) == 0:
        return candidates

    np.random.shuffle(opened_indices)
    np.random.shuffle(closed_indices)

    count = 0
    for j_open in opened_indices:
        for j_close in closed_indices[:4]:
            if count >= max_moves:
                break
            new_opened = facility_opened.copy()
            new_opened[j_open] = 0
            new_opened[j_close] = 1
            new_assignments = assign_clients_to_nearest_open_facility(
                service_costs, new_opened, facility_capacities, client_demands
            )
            if is_feasible(new_opened, new_assignments, facility_capacities, client_demands):
                move_key = ('swap', int(j_open), int(j_close))
                candidates.append((new_opened, new_assignments, move_key))
                count += 1
        if count >= max_moves:
            break

    return candidates


def generate_reassign_moves(facility_opened: np.ndarray, client_assignments: np.ndarray,
                             service_costs: np.ndarray, facility_capacities: np.ndarray,
                             client_demands: np.ndarray, max_moves: int = 20) -> List[Tuple]:
    """
    Generate move candidates by reassigning a client to a different open facility
    (Tabu Search move type 4).

    Parameters:
    -----------
    facility_opened : np.ndarray
        Binary array indicating which facilities are opened (shape: num_facilities)
    client_assignments : np.ndarray
        Array indicating the assigned facility for each client (shape: num_clients)
    service_costs : np.ndarray
        Service costs matrix (shape: num_clients × num_facilities)
    facility_capacities : np.ndarray
        Capacity of each facility (shape: num_facilities)
    client_demands : np.ndarray
        Demand of each client (shape: num_clients)
    max_moves : int
        Maximum number of candidate moves to generate

    Returns:
    --------
    candidates : List[Tuple]
        List of (new_facility_opened, new_client_assignments, move_key) tuples
    """
    candidates = []
    opened_indices = np.where(facility_opened == 1)[0]

    if len(opened_indices) <= 1:
        return candidates

    client_indices = list(range(len(client_assignments)))
    np.random.shuffle(client_indices)

    for i in client_indices:
        if len(candidates) >= max_moves:
            break
        current_fac = client_assignments[i]
        alternatives = opened_indices[opened_indices != current_fac]
        np.random.shuffle(alternatives)

        for j in alternatives[:3]:
            new_assignments = client_assignments.copy()
            new_assignments[i] = j
            if is_feasible(facility_opened, new_assignments, facility_capacities, client_demands):
                move_key = ('reassign', int(i), int(current_fac), int(j))
                candidates.append((facility_opened.copy(), new_assignments, move_key))
                break  # One candidate per client

    return candidates


def get_inverse_move(move_key: tuple) -> tuple:
    """
    Compute the inverse (reversal) of a Tabu Search move.
    The inverse is added to the tabu list to prevent undoing the move.

    Parameters:
    -----------
    move_key : tuple
        The move key (type, indices...)

    Returns:
    --------
    inverse_key : tuple
        The inverse move key
    """
    move_type = move_key[0]
    if move_type == 'open':
        return ('close', move_key[1])
    elif move_type == 'close':
        return ('open', move_key[1])
    elif move_type == 'swap':
        return ('swap', move_key[2], move_key[1])    # reverse the swap
    elif move_type == 'reassign':
        return ('reassign', move_key[1], move_key[3], move_key[2])   # swap old/new facility
    return move_key


print("✅ Tabu Search move generation modules loaded successfully.")


### 5.2 Tabu List and Candidate Selection

The **tabu list** is the core memory structure of Tabu Search. It stores the *inverse* of recently applied moves and forbids those inverse moves for a fixed number of iterations, called the **tabu tenure**.

- **Tabu tenure**: Controls how long a move stays forbidden (typically 5–15 iterations). A longer tenure increases diversification but may prevent useful moves.
- **Aspiration criterion**: A tabu move can be accepted if its resulting cost is strictly better than the best solution found so far (`cost < best_cost`). This prevents the tabu list from blocking globally improving moves.
- **Move selection**: At each iteration, *all* candidate moves (from all four move types) are evaluated. The best admissible move is selected (non-tabu, or tabu but satisfying aspiration).

The function `select_best_candidate` implements this logic by iterating over all generated candidates, checking tabu status, and applying the aspiration criterion.


In [ ]:
def generate_all_candidates(current_solution: Tuple[np.ndarray, np.ndarray],
                            service_costs: np.ndarray, facility_capacities: np.ndarray,
                            client_demands: np.ndarray,
                            max_candidates: int = 20) -> List[Tuple]:
    """
    Generate all candidate moves for the current solution by applying all four move types.

    Parameters:
    -----------
    current_solution : Tuple[np.ndarray, np.ndarray]
        Current solution as (facility_opened, client_assignments)
    service_costs : np.ndarray
        Service costs matrix (shape: num_clients × num_facilities)
    facility_capacities : np.ndarray
        Capacity of each facility (shape: num_facilities)
    client_demands : np.ndarray
        Demand of each client (shape: num_clients)
    max_candidates : int
        Maximum number of candidates per move type

    Returns:
    --------
    all_candidates : List[Tuple]
        Combined list of (new_facility_opened, new_client_assignments, move_key) tuples
    """
    facility_opened, client_assignments = current_solution

    all_candidates = []
    all_candidates.extend(generate_open_moves(
        facility_opened, service_costs, facility_capacities, client_demands, max_candidates))
    all_candidates.extend(generate_close_moves(
        facility_opened, service_costs, facility_capacities, client_demands, max_candidates))
    all_candidates.extend(generate_swap_moves(
        facility_opened, service_costs, facility_capacities, client_demands, max_candidates))
    all_candidates.extend(generate_reassign_moves(
        facility_opened, client_assignments, service_costs, facility_capacities,
        client_demands, max_candidates))

    return all_candidates


def select_best_candidate(candidates: List[Tuple], tabu_list: dict, iteration: int,
                          best_cost: float, fixed_costs: np.ndarray,
                          service_costs: np.ndarray) -> Tuple:
    """
    Select the best admissible candidate move using tabu status and aspiration criterion.

    A candidate is admissible if:
    - Its move key is NOT in the tabu list (non-tabu), OR
    - Its resulting cost is strictly less than the current best known cost (aspiration).

    Parameters:
    -----------
    candidates : List[Tuple]
        List of (new_facility_opened, new_client_assignments, move_key) tuples
    tabu_list : dict
        Dictionary mapping inverse_move_key -> expiry_iteration
    iteration : int
        Current iteration number
    best_cost : float
        Best cost found so far (used for aspiration criterion)
    fixed_costs : np.ndarray
        Fixed costs for each facility
    service_costs : np.ndarray
        Service costs matrix

    Returns:
    --------
    best_candidate : Tuple or None
        (new_facility_opened, new_client_assignments, move_key, cost) or None if no admissible move
    """
    best_selected = None
    best_selected_cost = float('inf')

    for new_opened, new_assignments, move_key in candidates:
        cost = compute_total_cost(fixed_costs, service_costs, new_opened, new_assignments)

        # Check tabu status using the inverse move key
        inverse_key = get_inverse_move(move_key)
        is_tabu = (inverse_key in tabu_list) and (tabu_list[inverse_key] > iteration)

        # Accept if non-tabu OR aspiration criterion satisfied
        if not is_tabu or cost < best_cost:
            if cost < best_selected_cost:
                best_selected_cost = cost
                best_selected = (new_opened, new_assignments, move_key, cost)

    return best_selected


print("✅ Tabu List candidate selection module loaded successfully.")


### 5.3 Initial Solution Construction

Tabu Search starts from an initial solution constructed by a **greedy heuristic**:
1. Begin with all facilities closed.
2. Iteratively open the facility that provides the largest reduction in total cost.
3. Assign all clients to their nearest feasible open facility.

This greedy solution is then passed directly to the Tabu Search main loop as the starting point. Unlike VNS, Tabu Search does not require a separate local search phase after initialization — it performs guided neighborhood exploration from the very first iteration using the tabu mechanism.


In [ ]:
def tabu_initial_solution(instance: dict) -> Tuple[np.ndarray, np.ndarray]:
    """
    Construct an initial solution using the greedy heuristic for Tabu Search.
    
    Delegates to the existing greedy_initial_solution function and returns
    the solution tuple (facility_opened, client_assignments).
    
    Parameters:
    -----------
    instance : dict
        CFLP instance dictionary with keys: num_clients, num_facilities,
        service_costs, facility_capacities, client_demands
    
    Returns:
    --------
    initial_solution : Tuple[np.ndarray, np.ndarray]
        (facility_opened, client_assignments)
    """
    return greedy_initial_solution(
        num_clients=instance["num_clients"],
        num_facilities=instance["num_facilities"],
        service_costs=instance["service_costs"],
        facility_capacities=instance["facility_capacities"],
        client_demands=instance["client_demands"]
    )

print("✅ Tabu Search initialization module loaded successfully.")


---

### 5.4 Tabu Search — MAIN ALGORITHM

The complete Tabu Search algorithm for CFLP orchestrates all components:

**Pseudocode:**
```
1. s ← GreedyInitialSolution()
2. s* ← s  (best solution found)
3. TabuList ← {}
4. iteration ← 0, no_improve ← 0

5. WHILE iteration < max_iterations AND no_improve < max_no_improve:
   a. Generate all candidate moves from s (open, close, swap, reassign)
   b. Evaluate cost of each candidate
   c. Select best admissible candidate s' (non-tabu OR aspiration satisfied)
   d. Apply the move: s ← s'
   e. Add inverse(move) to TabuList with expiry = iteration + tabu_tenure
   f. IF cost(s') < cost(s*):
          s* ← s'   (new best found)
          no_improve ← 0
      ELSE:
          no_improve ← no_improve + 1
   g. iteration ← iteration + 1

6. RETURN s*, trajectory
```

**Key parameters:**
- `max_iterations`: Maximum total iterations (default: 200)
- `tabu_tenure`: How many iterations a move stays forbidden (default: 10)
- `max_candidates`: Number of candidates generated per move type per iteration (default: 20)
- `max_no_improve`: Stop early if no improvement for this many consecutive iterations (default: 40)

**Important**: Tabu Search always moves to the best admissible neighbor, even if it is worse than the current solution. This allows escaping local optima. The tabu list prevents cycling.


In [ ]:
def tabu_search_solve_cflp(instance: dict,
                           max_iterations: int = 200,
                           tabu_tenure: int = 10,
                           max_candidates: int = 20,
                           max_no_improve: int = 40) -> Tuple[Tuple[np.ndarray, np.ndarray], Tuple[List[float], List[int]]]:
    """
    Tabu Search for the Capacitated Facility Location Problem (CFLP).

    At each iteration, all feasible move candidates (open, close, swap, reassign) are
    generated from the current solution. The best admissible move is selected — either
    non-tabu, or tabu but satisfying the aspiration criterion (improves best known cost).
    The inverse of the applied move is added to the tabu list to prevent immediate reversal.

    Parameters:
    -----------
    instance : dict
        CFLP instance dictionary with keys:
            num_facilities, num_clients, fixed_costs, service_costs,
            facility_capacities, client_demands
    max_iterations : int
        Maximum number of Tabu Search iterations (default: 200)
    tabu_tenure : int
        Number of iterations a move's inverse remains forbidden (default: 10)
    max_candidates : int
        Maximum candidates generated per move type per iteration (default: 20)
    max_no_improve : int
        Stop early if no improvement for this many consecutive iterations (default: 40)

    Returns:
    --------
    best_solution : Tuple[np.ndarray, np.ndarray]
        Best solution found as (facility_opened, client_assignments)
    trajectory_data : Tuple[List[float], List[int]]
        (trajectory, trajectory_iterations) — costs and iterations where improvements occurred
    """
    fixed_costs = instance["fixed_costs"]
    service_costs = instance["service_costs"]
    facility_capacities = instance["facility_capacities"]
    client_demands = instance["client_demands"]

    # ── Step 1: Greedy initial solution ──────────────────────────────────────
    current_solution = tabu_initial_solution(instance)
    current_cost = compute_total_cost(
        fixed_costs, service_costs, current_solution[0], current_solution[1]
    )

    best_solution = current_solution
    best_cost = current_cost
    trajectory = [best_cost]
    trajectory_iterations = [0]

    # ── Step 2: Tabu list (inverse_move_key → expiry iteration) ──────────────
    tabu_list: dict = {}

    no_improve_count = 0
    print(f"Starting Tabu Search for CFLP (tenure={tabu_tenure}, max_iter={max_iterations})...")

    # ── Step 3: Main loop ─────────────────────────────────────────────────────
    for iteration in range(1, max_iterations + 1):

        # 3a. Generate all candidate moves
        all_candidates = generate_all_candidates(
            current_solution, service_costs, facility_capacities, client_demands, max_candidates
        )

        if not all_candidates:
            print(f"  No candidates generated at iteration {iteration}. Stopping.")
            break

        # 3b. Select best admissible candidate (tabu status + aspiration)
        selected = select_best_candidate(
            all_candidates, tabu_list, iteration, best_cost, fixed_costs, service_costs
        )

        if selected is None:
            # All moves are tabu — extremely rare; skip iteration
            no_improve_count += 1
            if no_improve_count >= max_no_improve:
                break
            continue

        new_opened, new_assignments, move_key, new_cost = selected

        # 3c. Apply move: update current solution
        current_solution = (new_opened, new_assignments)
        current_cost = new_cost

        # 3d. Add inverse of the applied move to tabu list
        inverse_key = get_inverse_move(move_key)
        tabu_list[inverse_key] = iteration + tabu_tenure

        # Optional: clean up expired tabu entries to keep memory bounded
        tabu_list = {k: v for k, v in tabu_list.items() if v > iteration}

        # 3e. Update best solution
        if new_cost < best_cost:
            best_solution = current_solution
            best_cost = new_cost
            trajectory.append(best_cost)
            trajectory_iterations.append(iteration)
            no_improve_count = 0
        else:
            no_improve_count += 1

        if no_improve_count >= max_no_improve:
            break

    print(f"Tabu Search completed after {iteration} iterations.")
    print(f"Best cost: {best_cost:.2f}")

    return best_solution, (trajectory, trajectory_iterations)


print("✅ Tabu Search (TS) module loaded successfully.")

# Test Tabu Search (uncomment to run)
# best_solution, trajectory = tabu_search_solve_cflp(
#     instance=instance,
#     max_iterations=200,
#     tabu_tenure=10,
#     max_candidates=20,
#     max_no_improve=40
# )
# print(f"Final solution cost: {trajectory[0][-1]:.2f}")


---

## 6. Genetic Algorithm (GA) Implementation

In this section, we implement a **Genetic Algorithm (GA)** to solve the CFLP. GAs are population-based metaheuristics inspired by Darwinian natural evolution. Instead of a single trajectory, the GA evolves a *population* of candidate solutions through generations using biologically-inspired operators.

### 6.1 Chromosome Encoding and Population

**Encoding**: Each individual (chromosome) is a binary vector of length `num_facilities`:
- `chromosome[j] = 1` if facility `j` is open, `0` otherwise.
- Client assignments are derived deterministically from the chromosome using the greedy nearest-feasible-facility assignment.

**Population initialization**: Combines one greedy solution with `population_size - 1` random feasible solutions. Random individuals are repaired if infeasible.

### 6.2 Selection — Binary Tournament

Two individuals are drawn at random; the one with lower cost (higher fitness) wins and becomes a parent. This is repeated twice to select two parents per crossover.

### 6.3 Crossover — Uniform Crossover

Each gene position is independently swapped between parents with probability 0.5:
- For each facility index `j`, if `random() < 0.5`: child1[j] = parent2[j], child2[j] = parent1[j].
- This produces two children with mixed facility decisions from both parents.
- Applied with probability `crossover_rate` (default: 0.85); otherwise children are clones.

### 6.4 Mutation — Bit-Flip

Each gene in a chromosome is independently flipped (0→1 or 1→0) with probability `mutation_rate` (default: 0.1). This introduces random variation and prevents premature convergence.

### 6.5 Repair Operator

After crossover and mutation, a chromosome may be infeasible (e.g., all facilities closed, or total capacity insufficient). The **repair operator**:
1. If no facility is open → open the facility with the highest capacity.
2. Iteratively open the highest-capacity closed facility until the assignment becomes feasible.

### 6.6 Survivor Selection — Elitism + Generational Replacement

- **Elitism**: The top `elitism` individuals (default: 2) are copied unchanged to the next generation.
- **Generational replacement**: The rest of the next generation is filled from offspring, sorted by cost.

### 6.7 Stopping Criterion

The GA stops after `max_generations` generations or when no improvement is found for `max_no_improve` consecutive generations.


In [ ]:
# ==============================================================================
# 6.1 Chromosome Repair and Population Initialization
# ==============================================================================

def repair_chromosome(facility_opened: np.ndarray, service_costs: np.ndarray,
                      facility_capacities: np.ndarray, client_demands: np.ndarray
                      ) -> Tuple[np.ndarray, np.ndarray]:
    """
    Repair an infeasible chromosome by opening additional facilities until feasible.

    Parameters:
    -----------
    facility_opened : np.ndarray
        Binary facility decision vector (may be infeasible)
    service_costs : np.ndarray
        Service costs matrix (shape: num_clients × num_facilities)
    facility_capacities : np.ndarray
        Capacity of each facility
    client_demands : np.ndarray
        Demand of each client

    Returns:
    --------
    repaired_opened : np.ndarray
        Repaired binary facility decision vector (feasible)
    assignments : np.ndarray
        Client assignments derived from repaired_opened
    """
    repaired = facility_opened.copy()

    # Ensure at least one facility is open
    if repaired.sum() == 0:
        j = int(np.argmax(facility_capacities))
        repaired[j] = 1

    assignments = assign_clients_to_nearest_open_facility(
        service_costs, repaired, facility_capacities, client_demands
    )

    # Open more facilities if still infeasible (capacity insufficient)
    max_attempts = len(repaired)
    for _ in range(max_attempts):
        if is_feasible(repaired, assignments, facility_capacities, client_demands):
            break
        closed = np.where(repaired == 0)[0]
        if len(closed) == 0:
            break
        # Open the closed facility with the largest capacity
        j = closed[int(np.argmax(facility_capacities[closed]))]
        repaired[j] = 1
        assignments = assign_clients_to_nearest_open_facility(
            service_costs, repaired, facility_capacities, client_demands
        )

    return repaired, assignments


def initialize_ga_population(population_size: int, num_facilities: int,
                             service_costs: np.ndarray, facility_capacities: np.ndarray,
                             client_demands: np.ndarray, fixed_costs: np.ndarray,
                             num_clients: int) -> List[np.ndarray]:
    """
    Initialize a diverse population of chromosomes for the Genetic Algorithm.

    Includes one greedy solution and (population_size - 1) randomly generated,
    repaired individuals.

    Parameters:
    -----------
    population_size : int
        Number of individuals in the population
    num_facilities : int
        Number of potential facilities
    service_costs, facility_capacities, client_demands, fixed_costs : np.ndarray
        CFLP instance data arrays
    num_clients : int
        Number of clients

    Returns:
    --------
    population : List[np.ndarray]
        List of binary chromosome arrays (facility decision vectors)
    """
    population = []

    # Seed population with the greedy solution
    greedy_sol = greedy_initial_solution(
        num_clients=num_clients,
        num_facilities=num_facilities,
        service_costs=service_costs,
        facility_capacities=facility_capacities,
        client_demands=client_demands
    )
    population.append(greedy_sol[0].copy())

    # Add random feasible individuals
    attempts = 0
    while len(population) < population_size and attempts < population_size * 10:
        attempts += 1
        num_open = np.random.randint(1, num_facilities + 1)
        indices = np.random.choice(num_facilities, num_open, replace=False)
        chromosome = np.zeros(num_facilities, dtype=int)
        chromosome[indices] = 1
        repaired, _ = repair_chromosome(chromosome, service_costs, facility_capacities, client_demands)
        population.append(repaired)

    # Fill remaining slots with greedy clones if random generation failed
    while len(population) < population_size:
        population.append(greedy_sol[0].copy())

    return population


def evaluate_chromosome(chromosome: np.ndarray, fixed_costs: np.ndarray,
                         service_costs: np.ndarray, facility_capacities: np.ndarray,
                         client_demands: np.ndarray) -> float:
    """
    Evaluate the fitness (total cost) of a chromosome.

    Parameters:
    -----------
    chromosome : np.ndarray
        Binary facility decision vector
    fixed_costs, service_costs, facility_capacities, client_demands : np.ndarray
        CFLP instance data

    Returns:
    --------
    cost : float
        Total cost (inf if infeasible)
    """
    assignments = assign_clients_to_nearest_open_facility(
        service_costs, chromosome, facility_capacities, client_demands
    )
    if not is_feasible(chromosome, assignments, facility_capacities, client_demands):
        return float('inf')
    return compute_total_cost(fixed_costs, service_costs, chromosome, assignments)


# ==============================================================================
# 6.2 Selection, Crossover, and Mutation Operators
# ==============================================================================

def tournament_selection(population: List[np.ndarray], fitness: List[float],
                         tournament_size: int = 3) -> np.ndarray:
    """
    Select one individual using binary tournament selection.

    Randomly samples `tournament_size` individuals and returns the one with
    the lowest cost (best fitness).

    Parameters:
    -----------
    population : List[np.ndarray]
        Current population of chromosomes
    fitness : List[float]
        Corresponding fitness (cost) values
    tournament_size : int
        Number of individuals in each tournament (default: 3)

    Returns:
    --------
    winner : np.ndarray
        Copy of the winning chromosome
    """
    indices = np.random.choice(len(population), min(tournament_size, len(population)), replace=False)
    winner_idx = int(indices[np.argmin([fitness[i] for i in indices])])
    return population[winner_idx].copy()


def uniform_crossover(parent1: np.ndarray, parent2: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """
    Apply uniform crossover on two parent chromosomes.

    Each gene is independently inherited from either parent with equal probability (0.5).

    Parameters:
    -----------
    parent1, parent2 : np.ndarray
        Parent binary chromosomes

    Returns:
    --------
    child1, child2 : Tuple[np.ndarray, np.ndarray]
        Two offspring chromosomes
    """
    mask = np.random.random(len(parent1)) < 0.5
    child1 = parent1.copy()
    child2 = parent2.copy()
    child1[mask] = parent2[mask]
    child2[mask] = parent1[mask]
    return child1, child2


def mutate_chromosome(chromosome: np.ndarray, mutation_rate: float) -> np.ndarray:
    """
    Apply bit-flip mutation to a chromosome.

    Each gene is independently flipped (0→1 or 1→0) with probability `mutation_rate`.

    Parameters:
    -----------
    chromosome : np.ndarray
        Binary chromosome
    mutation_rate : float
        Probability of flipping each gene (default: 0.1)

    Returns:
    --------
    mutated : np.ndarray
        Mutated binary chromosome
    """
    mutated = chromosome.copy()
    flip_mask = np.random.random(len(mutated)) < mutation_rate
    mutated[flip_mask] = 1 - mutated[flip_mask]
    return mutated


# ==============================================================================
# 6.3 Genetic Algorithm — Main Algorithm
# ==============================================================================

def genetic_algorithm_solve_cflp(instance: dict,
                                  population_size: int = 30,
                                  max_generations: int = 100,
                                  crossover_rate: float = 0.85,
                                  mutation_rate: float = 0.10,
                                  tournament_size: int = 3,
                                  elitism: int = 2,
                                  max_no_improve: int = 30
                                  ) -> Tuple[Tuple[np.ndarray, np.ndarray], Tuple[List[float], List[int]]]:
    """
    Genetic Algorithm for the Capacitated Facility Location Problem (CFLP).

    Evolves a population of binary chromosomes (facility open/close decisions) through
    selection, uniform crossover, bit-flip mutation, feasibility repair, and elitist
    generational replacement.

    Parameters:
    -----------
    instance : dict
        CFLP instance dictionary with keys:
            num_facilities, num_clients, fixed_costs, service_costs,
            facility_capacities, client_demands
    population_size : int
        Number of individuals per generation (default: 30)
    max_generations : int
        Maximum number of generations (default: 100)
    crossover_rate : float
        Probability of applying crossover to a pair of parents (default: 0.85)
    mutation_rate : float
        Per-gene flip probability during mutation (default: 0.10)
    tournament_size : int
        Number of contestants in each tournament selection (default: 3)
    elitism : int
        Number of best individuals carried unchanged to the next generation (default: 2)
    max_no_improve : int
        Stop early if no improvement found for this many consecutive generations (default: 30)

    Returns:
    --------
    best_solution : Tuple[np.ndarray, np.ndarray]
        Best solution found as (facility_opened, client_assignments)
    trajectory_data : Tuple[List[float], List[int]]
        (trajectory, trajectory_generations) — costs and generations where improvements occurred
    """
    num_facilities = instance["num_facilities"]
    num_clients = instance["num_clients"]
    fixed_costs = instance["fixed_costs"]
    service_costs = instance["service_costs"]
    facility_capacities = instance["facility_capacities"]
    client_demands = instance["client_demands"]

    # ── Step 1: Initialize population ────────────────────────────────────────
    population = initialize_ga_population(
        population_size, num_facilities, service_costs,
        facility_capacities, client_demands, fixed_costs, num_clients
    )
    fitness = [evaluate_chromosome(ind, fixed_costs, service_costs,
                                   facility_capacities, client_demands)
               for ind in population]

    best_idx = int(np.argmin(fitness))
    best_fitness = fitness[best_idx]
    best_chromosome = population[best_idx].copy()

    trajectory = [best_fitness]
    trajectory_generations = [0]
    no_improve_count = 0

    print(f"Starting Genetic Algorithm for CFLP "
          f"(pop={population_size}, max_gen={max_generations})...")

    # ── Step 2: Evolutionary loop ─────────────────────────────────────────────
    for generation in range(1, max_generations + 1):

        # Elitism: preserve top individuals unchanged
        sorted_indices = np.argsort(fitness)
        elites = [population[i].copy() for i in sorted_indices[:elitism]]
        elite_fitness = [fitness[i] for i in sorted_indices[:elitism]]

        # Generate offspring to fill remaining slots
        offspring = []
        offspring_fitness = []

        while len(offspring) < population_size - elitism:
            # Selection
            p1 = tournament_selection(population, fitness, tournament_size)
            p2 = tournament_selection(population, fitness, tournament_size)

            # Crossover
            if np.random.random() < crossover_rate:
                c1, c2 = uniform_crossover(p1, p2)
            else:
                c1, c2 = p1.copy(), p2.copy()

            # Mutation
            c1 = mutate_chromosome(c1, mutation_rate)
            c2 = mutate_chromosome(c2, mutation_rate)

            # Repair infeasible offspring
            c1, _ = repair_chromosome(c1, service_costs, facility_capacities, client_demands)
            c2, _ = repair_chromosome(c2, service_costs, facility_capacities, client_demands)

            # Evaluate
            f1 = evaluate_chromosome(c1, fixed_costs, service_costs,
                                     facility_capacities, client_demands)
            f2 = evaluate_chromosome(c2, fixed_costs, service_costs,
                                     facility_capacities, client_demands)

            offspring.extend([c1, c2])
            offspring_fitness.extend([f1, f2])

        # Trim to exact size needed
        offspring = offspring[:population_size - elitism]
        offspring_fitness = offspring_fitness[:population_size - elitism]

        # New population: elites + offspring
        population = elites + offspring
        fitness = elite_fitness + offspring_fitness

        # Update best solution
        current_best_idx = int(np.argmin(fitness))
        if fitness[current_best_idx] < best_fitness:
            best_fitness = fitness[current_best_idx]
            best_chromosome = population[current_best_idx].copy()
            trajectory.append(best_fitness)
            trajectory_generations.append(generation)
            no_improve_count = 0
        else:
            no_improve_count += 1

        if no_improve_count >= max_no_improve:
            break

    # ── Step 3: Derive final solution from best chromosome ────────────────────
    best_assignments = assign_clients_to_nearest_open_facility(
        service_costs, best_chromosome, facility_capacities, client_demands
    )
    best_solution = (best_chromosome, best_assignments)

    print(f"Genetic Algorithm completed after {generation} generations.")
    print(f"Best cost: {best_fitness:.2f}")

    return best_solution, (trajectory, trajectory_generations)


print("✅ Genetic Algorithm (GA) module loaded successfully.")

# Test GA (uncomment to run)
# best_solution, (trajectory, traj_gen) = genetic_algorithm_solve_cflp(
#     instance=instance,
#     population_size=30,
#     max_generations=100
# )
# print(f"GA best cost: {trajectory[-1]:.2f} after {traj_gen[-1]} generations")


---

## 7. Exact Solution using Gurobi

In this section, we implement the exact solution for the Capacitated Facility Location Problem (CFLP) using the Gurobi optimization solver. Gurobi solves the MIP formulation to **guaranteed optimality** (within the given time limit) using Branch-and-Bound with cutting planes.

**Role in this assignment**: Gurobi provides the **optimal reference solution** against which both Tabu Search and Genetic Algorithm are benchmarked. The **optimality gap** for each metaheuristic is computed as:

$$\text{Gap}(\%) = \frac{\text{TS/GA cost} - \text{Gurobi cost}}{\text{Gurobi cost}} \times 100$$


In [ ]:
def gurobi_solve_cflp(instance: dict, time_limit: int = 300, check_size_limit: bool = False) -> Tuple[np.ndarray, np.ndarray, float]:
    """
    Solve the Capacitated Facility Location Problem (CFLP) using Gurobi optimization solver.
    
    Parameters:
    -----------
    instance : dict
        Dictionary containing the CFLP instance data with keys:
            - num_facilities
            - num_clients
            - fixed_costs
            - service_costs
            - facility_capacities
            - client_demands
    time_limit : int
        Time limit for the solver in seconds (default: 300 seconds)
    check_size_limit : bool
        If True, check if instance exceeds free Gurobi limits (2000 vars, 2000 constraints)
        and skip solving if it does (default: False for academic/unlimited version)
        
    Returns:
    --------
    facility_opened : np.ndarray
        Binary array indicating which facilities are opened (shape: num_facilities)
    client_assignments : np.ndarray
        Array indicating the assigned facility for each client (shape: num_clients)
    optimal_cost : float
        Optimal total cost of the solution
        
    Raises:
    -------
    ValueError
        If instance exceeds free Gurobi limits (when check_size_limit=True)
    """
    from gurobipy import Model, GRB, quicksum

    num_facilities = instance["num_facilities"]
    num_clients = instance["num_clients"]
    fixed_costs = instance["fixed_costs"]
    service_costs = instance["service_costs"]
    facility_capacities = instance["facility_capacities"]
    client_demands = instance["client_demands"]

    # Check size limits for free Gurobi version
    if check_size_limit:
        num_variables = num_facilities + (num_clients * num_facilities)  # y + x variables
        num_constraints = num_clients + (num_clients * num_facilities) + num_facilities  # assignment + linking + capacity
        
        if num_variables > 2000 or num_constraints > 2000:
            raise ValueError(
                f"Instance exceeds free Gurobi limits: "
                f"{num_variables} variables (max 2000), "
                f"{num_constraints} constraints (max 2000). "
                f"Skipping this instance."
            )

    # Create Gurobi model
    model = Model("Capacitated_Facility_Location_Problem")
    model.setParam('OutputFlag', 0)  # Suppress output
    model.setParam('TimeLimit', time_limit)
    
    # Decision variables
    y = model.addVars(num_facilities, vtype=GRB.BINARY, name="y")  # Facility open variables
    x = model.addVars(num_clients, num_facilities, vtype=GRB.BINARY, name="x")  # Client assignment variables
    
    # Objective function: Minimize total cost
    model.setObjective(
        quicksum(fixed_costs[j] * y[j] for j in range(num_facilities)) +
        quicksum(service_costs[i][j] * x[i, j] for i in range(num_clients) for j in range(num_facilities)),
        GRB.MINIMIZE
    )
    
    # Constraints: Each client must be assigned to exactly one facility
    for i in range(num_clients):
        model.addConstr(quicksum(x[i, j] for j in range(num_facilities)) == 1, name=f"Client_{i}_assignment")
    
    # Constraints: Clients can only be assigned to opened facilities
    for i in range(num_clients):
        for j in range(num_facilities):
            model.addConstr(x[i, j] <= y[j], name=f"Client_{i}_to_Facility_{j}_open")
    
    # NEW: Capacity constraints for CFLP
    for j in range(num_facilities):
        model.addConstr(
            quicksum(client_demands[i] * x[i, j] for i in range(num_clients)) <= facility_capacities[j] * y[j],
            name=f"Facility_{j}_capacity"
        )
    
    # Optimize
    model.optimize()

    # Check if solution was found
    if model.status == GRB.OPTIMAL or model.status == GRB.TIME_LIMIT:
        # Extract solution
        facility_opened = np.array([int(round(y[j].X)) for j in range(num_facilities)])
        client_assignments = np.array([int(round(sum(j * x[i, j].X for j in range(num_facilities)))) 
                                        for i in range(num_clients)])
        optimal_cost = model.objVal
        
        return facility_opened, client_assignments, optimal_cost
    else:
        raise Exception(f"Gurobi optimization failed with status: {model.status}")

print("✅ Gurobi module loaded successfully.")

# Test Gurobi (uncomment the following lines to run a test)
# gurobi_facilities, gurobi_assignments, gurobi_cost = gurobi_solve_cflp(instance, time_limit=60)
# print(f"Gurobi optimal cost: {gurobi_cost:.2f}")
# print(f"Facilities opened: {gurobi_facilities}")
# print(f"Number of facilities opened: {gurobi_facilities.sum()}")

In [ ]:
def compare_solutions(instance: dict,
                      gurobi_solution: Tuple[np.ndarray, np.ndarray, float],
                      ts_solution: Tuple[Tuple[np.ndarray, np.ndarray], Tuple[List[float], List[int]]],
                      ga_solution: Tuple[Tuple[np.ndarray, np.ndarray], Tuple[List[float], List[int]]]):
    """
    Compare Gurobi, Tabu Search, and Genetic Algorithm solutions side by side.

    Parameters:
    -----------
    instance : dict
        CFLP instance dictionary
    gurobi_solution : Tuple[np.ndarray, np.ndarray, float]
        Gurobi result as (facility_opened, client_assignments, optimal_cost)
    ts_solution : Tuple[Tuple[np.ndarray, np.ndarray], Tuple[List[float], List[int]]]
        TS result as ((facility_opened, client_assignments), (trajectory, trajectory_iterations))
    ga_solution : Tuple[Tuple[np.ndarray, np.ndarray], Tuple[List[float], List[int]]]
        GA result as ((facility_opened, client_assignments), (trajectory, trajectory_generations))
    """
    fixed_costs = instance['fixed_costs']
    service_costs = instance['service_costs']

    gurobi_facilities, gurobi_assignments, gurobi_cost = gurobi_solution

    (ts_facilities, ts_assignments), (ts_traj, ts_traj_iter) = ts_solution
    ts_cost = compute_total_cost(fixed_costs, service_costs, ts_facilities, ts_assignments)
    ts_gap = ((ts_cost - gurobi_cost) / gurobi_cost) * 100
    ts_total_iters = ts_traj_iter[-1] if ts_traj_iter else 0
    ts_improvements = len(ts_traj) - 1

    (ga_facilities, ga_assignments), (ga_traj, ga_traj_gen) = ga_solution
    ga_cost = compute_total_cost(fixed_costs, service_costs, ga_facilities, ga_assignments)
    ga_gap = ((ga_cost - gurobi_cost) / gurobi_cost) * 100
    ga_total_gens = ga_traj_gen[-1] if ga_traj_gen else 0
    ga_improvements = len(ga_traj) - 1

    print("=" * 70)
    print("SOLUTION COMPARISON: Gurobi vs Tabu Search vs Genetic Algorithm")
    print("=" * 70)

    print(f"\n{'Method':<20} {'Cost':>14} {'Facilities':>10} {'Gap (%)':>10}")
    print("-" * 56)
    print(f"{'Gurobi (Exact)':<20} {gurobi_cost:>14.2f} {int(gurobi_facilities.sum()):>10} {'0.00':>10}")
    print(f"{'Tabu Search':<20} {ts_cost:>14.2f} {int(ts_facilities.sum()):>10} {ts_gap:>10.2f}")
    print(f"{'Genetic Algorithm':<20} {ga_cost:>14.2f} {int(ga_facilities.sum()):>10} {ga_gap:>10.2f}")

    print(f"\nTabu Search details: {ts_total_iters} iterations, {ts_improvements} improvements found")
    print(f"Genetic Algorithm details: {ga_total_gens} generations, {ga_improvements} improvements found")
    print("=" * 70)


print("✅ Comparison module loaded successfully.")


---

## 8. Experiments and Comparison

In this section, we conduct a series of experiments to evaluate and compare the performance of three approaches for solving the CFLP:

1. **Gurobi** (exact MIP solver) — provides optimal solutions as ground truth.
2. **Tabu Search (TS)** — trajectory-based metaheuristic using a tabu list and aspiration criterion.
3. **Genetic Algorithm (GA)** — population-based metaheuristic using selection, crossover, mutation, and repair.

Experiments assess:
- **Solution quality**: Optimality gap (%) relative to Gurobi for both TS and GA.
- **Computational time**: Solver time for each of the three methods.
- **Convergence behavior**: How quickly TS and GA improve over iterations/generations.
- **Scalability**: How each method's performance changes across small, medium, and large instances.
- **Robustness**: Consistency across different difficulty levels (easy, moderate, expensive).


In [ ]:
def plot_convergence_ts_ga(ts_trajectory: List[float], ts_iterations: List[int],
                           ga_trajectory: List[float], ga_generations: List[int],
                           optimal_cost: float, instance_name: str,
                           output_folder: str, size: str, difficulty: str):
    """
    Plot and save the convergence trajectories of Tabu Search and Genetic Algorithm
    for a single instance, with the Gurobi optimal cost as reference.

    Parameters:
    -----------
    ts_trajectory, ts_iterations : Lists
        TS best costs and the iterations where improvements occurred
    ga_trajectory, ga_generations : Lists
        GA best costs and the generations where improvements occurred
    optimal_cost : float
        Gurobi optimal cost (reference line)
    instance_name, output_folder, size, difficulty : str
        Instance metadata and output path
    """
    plt.figure(figsize=(14, 8))
    plt.style.use('seaborn-v0_8-darkgrid')

    def make_step_curve(iterations, costs):
        """Build a step-wise (staircase) representation of the trajectory."""
        xi, yi = [], []
        for i in range(len(iterations)):
            xi.append(iterations[i])
            yi.append(costs[i])
            if i < len(iterations) - 1:
                xi.append(iterations[i + 1])
                yi.append(costs[i])
        return np.array(xi), np.array(yi)

    # ── TS trajectory ────────────────────────────────────────────────────────
    ts_xi, ts_yi = make_step_curve(ts_iterations, ts_trajectory)
    plt.plot(ts_xi, ts_yi, color='#2E86AB', linewidth=2.5, alpha=0.9,
             label='Tabu Search (TS)', zorder=2)
    plt.scatter(ts_iterations, ts_trajectory, s=80, color='#2E86AB',
                marker='o', edgecolors='white', linewidths=1.5, zorder=3)

    # ── GA trajectory ────────────────────────────────────────────────────────
    ga_xi, ga_yi = make_step_curve(ga_generations, ga_trajectory)
    plt.plot(ga_xi, ga_yi, color='#E07B39', linewidth=2.5, alpha=0.9,
             label='Genetic Algorithm (GA)', zorder=2)
    plt.scatter(ga_generations, ga_trajectory, s=80, color='#E07B39',
                marker='s', edgecolors='white', linewidths=1.5, zorder=3)

    # ── Gurobi reference ─────────────────────────────────────────────────────
    plt.axhline(y=optimal_cost, color='#C73E1D', linestyle='--', linewidth=2.5,
                alpha=0.85, label=f'Gurobi Optimal: {optimal_cost:.2f}', zorder=1)

    # Final gaps
    ts_final_gap = ((ts_trajectory[-1] - optimal_cost) / optimal_cost) * 100
    ga_final_gap = ((ga_trajectory[-1] - optimal_cost) / optimal_cost) * 100

    plt.title(f'Convergence: {instance_name}\n'
              f'Size: {size.capitalize()} | Difficulty: {difficulty.capitalize()} | '
              f'TS Gap: {ts_final_gap:.2f}% | GA Gap: {ga_final_gap:.2f}%',
              fontsize=13, fontweight='bold', pad=15)
    plt.xlabel('Iteration / Generation', fontsize=12, fontweight='bold')
    plt.ylabel('Total Cost', fontsize=12, fontweight='bold')
    plt.legend(loc='upper right', fontsize=11, framealpha=0.95, shadow=True,
               fancybox=True, borderpad=1)
    plt.grid(True, alpha=0.3)

    ax = plt.gca()
    ax.set_facecolor('#FAFAFA')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # Stats box
    textstr = (f'TS:  {ts_trajectory[0]:>10,.2f} → {ts_trajectory[-1]:>10,.2f}  '
               f'(gap {ts_final_gap:.2f}%)\n'
               f'GA:  {ga_trajectory[0]:>10,.2f} → {ga_trajectory[-1]:>10,.2f}  '
               f'(gap {ga_final_gap:.2f}%)')
    props = dict(boxstyle='round,pad=0.6', facecolor='#F0F0F0',
                 edgecolor='#2E86AB', linewidth=1.5, alpha=0.95)
    plt.text(0.02, 0.05, textstr, transform=ax.transAxes, fontsize=9.5,
             verticalalignment='bottom', bbox=props, family='monospace')

    plt.tight_layout()
    plot_path = os.path.join(output_folder,
                             instance_name.replace('.json', '_convergence.png'))
    plt.savefig(plot_path, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close()


def run_experiments(instances_folder: str = "cflp_instances",
                    results_file: str = "results_comparison.csv",
                    ts_max_iterations: int = 200,
                    ts_tabu_tenure: int = 10,
                    ts_max_candidates: int = 20,
                    ts_max_no_improve: int = 40,
                    ga_population_size: int = 30,
                    ga_max_generations: int = 100,
                    ga_crossover_rate: float = 0.85,
                    ga_mutation_rate: float = 0.10,
                    ga_max_no_improve: int = 30,
                    gurobi_time_limit: int = 300,
                    save_convergence_plots: bool = True,
                    convergence_plots_folder: str = "convergence_plots",
                    gurobi_version: str = "academic"):
    """
    Run experiments comparing Gurobi, Tabu Search, and Genetic Algorithm
    on all CFLP instances in the specified folder.

    Parameters:
    -----------
    instances_folder : str
        Folder containing JSON instance files
    results_file : str
        CSV file for saving results
    ts_* : Tabu Search hyperparameters
    ga_* : Genetic Algorithm hyperparameters
    gurobi_time_limit : int
        Time limit (seconds) for Gurobi
    save_convergence_plots : bool
        Whether to save convergence plots (TS + GA vs Gurobi)
    convergence_plots_folder : str
        Folder for convergence plot output
    gurobi_version : str
        "academic" (unlimited) or "free" (max 2000 vars/constraints)

    Returns:
    --------
    results_df : pd.DataFrame
        DataFrame with results for all instances
    """
    if save_convergence_plots:
        os.makedirs(convergence_plots_folder, exist_ok=True)

    check_size_limit = (gurobi_version.lower() == "free")
    instance_files = sorted([f for f in os.listdir(instances_folder) if f.endswith('.json')])

    results = []
    skipped_instances = 0

    print(f"Running experiments on {len(instance_files)} instances...")
    print(f"Gurobi version: {gurobi_version.upper()}")
    if check_size_limit:
        print("⚠️  Free version: instances exceeding 2000 vars/constraints skipped for Gurobi.\n")
    else:
        print("✓ Academic version: no size limits.\n")

    for idx, instance_file in enumerate(instance_files, 1):
        instance_path = os.path.join(instances_folder, instance_file)
        print(f"[{idx}/{len(instance_files)}] Processing {instance_file}")

        # Load instance
        num_facilities, num_clients, fixed_costs, service_costs, \
            facility_capacities, client_demands = load_cflp_instance(instance_path)
        instance = {
            "num_facilities": num_facilities,
            "num_clients": num_clients,
            "fixed_costs": fixed_costs,
            "service_costs": service_costs,
            "facility_capacities": facility_capacities,
            "client_demands": client_demands
        }

        # Extract metadata from filename
        parts = instance_file.replace('.json', '').split('_')
        instance_id = int(parts[0])
        size = parts[2]
        difficulty = parts[3]

        # ── Gurobi ───────────────────────────────────────────────────────────
        print("  - Running Gurobi...")
        try:
            g_start = time.time()
            gurobi_facilities, gurobi_assignments, gurobi_cost = gurobi_solve_cflp(
                instance, time_limit=gurobi_time_limit, check_size_limit=check_size_limit
            )
            gurobi_time = time.time() - g_start
            gurobi_status = "Optimal"
        except ValueError:
            print("    ⚠️  Skipped (exceeds free version limits)")
            gurobi_cost = np.nan
            gurobi_time = np.nan
            gurobi_status = "Skipped (Size Limit)"
            skipped_instances += 1
        except Exception as e:
            print(f"    Gurobi failed: {e}")
            gurobi_cost = np.inf
            gurobi_time = gurobi_time_limit
            gurobi_status = "Failed"

        # ── Tabu Search ──────────────────────────────────────────────────────
        print("  - Running Tabu Search...")
        try:
            ts_start = time.time()
            ts_solution, (ts_traj, ts_traj_iter) = tabu_search_solve_cflp(
                instance,
                max_iterations=ts_max_iterations,
                tabu_tenure=ts_tabu_tenure,
                max_candidates=ts_max_candidates,
                max_no_improve=ts_max_no_improve
            )
            ts_time = time.time() - ts_start
            ts_facilities, ts_assignments = ts_solution
            ts_cost = compute_total_cost(fixed_costs, service_costs, ts_facilities, ts_assignments)
            ts_status = "Completed"
            ts_iterations = ts_traj_iter[-1] if ts_traj_iter else 0
        except Exception as e:
            print(f"    TS failed: {e}")
            ts_cost = np.inf
            ts_time = 0.0
            ts_status = "Failed"
            ts_iterations = 0
            ts_traj, ts_traj_iter = [], []

        # ── Genetic Algorithm ────────────────────────────────────────────────
        print("  - Running Genetic Algorithm...")
        try:
            ga_start = time.time()
            ga_solution, (ga_traj, ga_traj_gen) = genetic_algorithm_solve_cflp(
                instance,
                population_size=ga_population_size,
                max_generations=ga_max_generations,
                crossover_rate=ga_crossover_rate,
                mutation_rate=ga_mutation_rate,
                max_no_improve=ga_max_no_improve
            )
            ga_time = time.time() - ga_start
            ga_facilities, ga_assignments = ga_solution
            ga_cost = compute_total_cost(fixed_costs, service_costs, ga_facilities, ga_assignments)
            ga_status = "Completed"
            ga_generations_run = ga_traj_gen[-1] if ga_traj_gen else 0
        except Exception as e:
            print(f"    GA failed: {e}")
            ga_cost = np.inf
            ga_time = 0.0
            ga_status = "Failed"
            ga_generations_run = 0
            ga_traj, ga_traj_gen = [], []

        # ── Compute gaps ─────────────────────────────────────────────────────
        if gurobi_status == "Optimal" and gurobi_cost > 0 and not np.isnan(gurobi_cost):
            ts_gap = ((ts_cost - gurobi_cost) / gurobi_cost) * 100
            ga_gap = ((ga_cost - gurobi_cost) / gurobi_cost) * 100
        else:
            ts_gap = np.nan
            ga_gap = np.nan

        ts_speedup = gurobi_time / ts_time if ts_time > 0 and not np.isnan(gurobi_time) else np.nan
        ga_speedup = gurobi_time / ga_time if ga_time > 0 and not np.isnan(gurobi_time) else np.nan

        # Save convergence plot
        if (save_convergence_plots and gurobi_status == "Optimal"
                and ts_status == "Completed" and ga_status == "Completed"
                and ts_traj and ga_traj):
            try:
                plot_convergence_ts_ga(
                    ts_traj, ts_traj_iter, ga_traj, ga_traj_gen,
                    gurobi_cost, instance_file, convergence_plots_folder, size, difficulty
                )
            except Exception:
                pass

        # ── Store results ─────────────────────────────────────────────────────
        results.append({
            'instance_id': instance_id,
            'instance_name': instance_file,
            'size': size,
            'difficulty': difficulty,
            'num_facilities': num_facilities,
            'num_clients': num_clients,
            # Gurobi
            'gurobi_cost': gurobi_cost,
            'gurobi_time': gurobi_time,
            'gurobi_status': gurobi_status,
            # Tabu Search
            'ts_cost': ts_cost,
            'ts_time': ts_time,
            'ts_iterations': ts_iterations,
            'ts_status': ts_status,
            'ts_gap_percent': ts_gap,
            'ts_speedup_factor': ts_speedup,
            # Genetic Algorithm
            'ga_cost': ga_cost,
            'ga_time': ga_time,
            'ga_generations': ga_generations_run,
            'ga_status': ga_status,
            'ga_gap_percent': ga_gap,
            'ga_speedup_factor': ga_speedup,
        })

        print(f"  - Gurobi: {gurobi_cost:.2f} ({gurobi_time:.2f}s) | "
              f"TS: gap={ts_gap:.2f}% ({ts_time:.2f}s) | "
              f"GA: gap={ga_gap:.2f}% ({ga_time:.2f}s)\n")

    # ── Save results ──────────────────────────────────────────────────────────
    results_df = pd.DataFrame(results)
    results_df.to_csv(results_file, index=False)
    print(f"\nResults saved to {results_file}")

    xlsx_file = results_file.replace('.csv', '.xlsx')
    with pd.ExcelWriter(xlsx_file, engine='openpyxl') as writer:
        results_df.to_excel(writer, sheet_name='All Results', index=False)

        for col_prefix, label in [('ts', 'Tabu Search'), ('ga', 'Genetic Algorithm')]:
            for group_col, sheet_suffix in [('size', 'by Size'), ('difficulty', 'by Difficulty')]:
                summary = results_df.groupby(group_col).agg({
                    'gurobi_cost': ['mean', 'std'],
                    f'{col_prefix}_cost': ['mean', 'std'],
                    f'{col_prefix}_gap_percent': ['mean', 'min', 'max'],
                    'gurobi_time': ['mean', 'std'],
                    f'{col_prefix}_time': ['mean', 'std'],
                    f'{col_prefix}_speedup_factor': ['mean', 'std'],
                    'instance_id': 'count'
                }).round(2)
                summary.to_excel(writer, sheet_name=f'{label[:2]} {sheet_suffix}')

        for col_prefix, label in [('ts', 'TS'), ('ga', 'GA')]:
            pivot = results_df.pivot_table(
                values=f'{col_prefix}_gap_percent', index='size',
                columns='difficulty', aggfunc='mean'
            ).round(2)
            pivot.to_excel(writer, sheet_name=f'{label} Gap Matrix')

    print(f"Excel results saved to {xlsx_file}")

    # Summary
    valid = results_df[
        (results_df['gurobi_status'] == 'Optimal') &
        (results_df['ts_status'] == 'Completed') &
        (results_df['ga_status'] == 'Completed')
    ]
    print("\n" + "=" * 70)
    print("EXPERIMENTS SUMMARY")
    print("=" * 70)
    print(f"Total instances: {len(instance_files)}")
    print(f"Solved by all three methods: {len(valid)}")
    if len(valid) > 0:
        print(f"\nTabu Search  — avg gap: {valid['ts_gap_percent'].mean():.2f}%, "
              f"avg speedup: {valid['ts_speedup_factor'].mean():.2f}x")
        print(f"Genetic Algo — avg gap: {valid['ga_gap_percent'].mean():.2f}%, "
              f"avg speedup: {valid['ga_speedup_factor'].mean():.2f}x")
    print("=" * 70)

    return results_df


print("✅ Experiment runner (TS + GA + Gurobi) loaded successfully.")


---

## 9. Visualization and Analysis

This section provides comprehensive visualization tools for comparing Gurobi, Tabu Search, and Genetic Algorithm results across all instances.

**Plots generated:**
- **Optimality Gap (%)** — Boxplots of TS and GA gap by instance size and difficulty.
- **Computation Time** — Bar charts comparing Gurobi, TS, and GA runtimes.
- **Speedup Factor** — TS speedup and GA speedup over Gurobi (× faster).
- **Solution Cost Heatmaps** — Mean gap organized in a size × difficulty matrix for both TS and GA.
- **TS vs GA head-to-head** — Scatter plot comparing TS gap against GA gap per instance.


In [ ]:
def visualize_results(results_df: pd.DataFrame,
                      output_folder: str = "visualization_plots"):
    """
    Generate comprehensive comparison plots for Gurobi, Tabu Search, and Genetic Algorithm.

    Parameters:
    -----------
    results_df : pd.DataFrame
        DataFrame produced by run_experiments()
    output_folder : str
        Folder for saving plots
    """
    os.makedirs(output_folder, exist_ok=True)

    # Filter to instances solved by all three methods
    valid = results_df[
        (results_df['gurobi_status'] == 'Optimal') &
        (results_df['ts_status'] == 'Completed') &
        (results_df['ga_status'] == 'Completed')
    ].copy()

    if valid.empty:
        print("⚠️  No valid results to visualize.")
        return

    plt.style.use('seaborn-v0_8-darkgrid')
    SIZE_ORDER = ['small', 'medium', 'large']
    DIFF_ORDER = ['easy', 'moderate', 'expensive']
    TS_COLOR   = '#2E86AB'
    GA_COLOR   = '#E07B39'
    GRB_COLOR  = '#C73E1D'

    # ── 1. Optimality Gap by Instance Size ───────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=False)
    for ax, col, label, color in zip(
            axes,
            ['ts_gap_percent', 'ga_gap_percent'],
            ['Tabu Search', 'Genetic Algorithm'],
            [TS_COLOR, GA_COLOR]):
        sizes_present = [s for s in SIZE_ORDER if s in valid['size'].values]
        data_by_size = [valid[valid['size'] == s][col].dropna().values for s in sizes_present]
        bp = ax.boxplot(data_by_size, labels=[s.capitalize() for s in sizes_present],
                        patch_artist=True, widths=0.5, notch=False)
        for patch in bp['boxes']:
            patch.set_facecolor(color)
            patch.set_alpha(0.6)
        for median in bp['medians']:
            median.set_color('black')
            median.set_linewidth(2)
        ax.set_title(f'{label}: Optimality Gap by Size', fontsize=13, fontweight='bold')
        ax.set_xlabel('Instance Size', fontsize=11)
        ax.set_ylabel('Optimality Gap (%)', fontsize=11)
        ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(os.path.join(output_folder, 'gap_by_size.png'), dpi=150, bbox_inches='tight')
    plt.close()

    # ── 2. Optimality Gap by Difficulty ──────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=False)
    for ax, col, label, color in zip(
            axes,
            ['ts_gap_percent', 'ga_gap_percent'],
            ['Tabu Search', 'Genetic Algorithm'],
            [TS_COLOR, GA_COLOR]):
        diffs_present = [d for d in DIFF_ORDER if d in valid['difficulty'].values]
        data_by_diff = [valid[valid['difficulty'] == d][col].dropna().values for d in diffs_present]
        bp = ax.boxplot(data_by_diff, labels=[d.capitalize() for d in diffs_present],
                        patch_artist=True, widths=0.5)
        for patch in bp['boxes']:
            patch.set_facecolor(color)
            patch.set_alpha(0.6)
        for median in bp['medians']:
            median.set_color('black')
            median.set_linewidth(2)
        ax.set_title(f'{label}: Optimality Gap by Difficulty', fontsize=13, fontweight='bold')
        ax.set_xlabel('Difficulty', fontsize=11)
        ax.set_ylabel('Optimality Gap (%)', fontsize=11)
        ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(os.path.join(output_folder, 'gap_by_difficulty.png'), dpi=150, bbox_inches='tight')
    plt.close()

    # ── 3. Computation Time Comparison ───────────────────────────────────────
    fig, ax = plt.subplots(figsize=(14, 6))
    sizes_present = [s for s in SIZE_ORDER if s in valid['size'].values]
    x = np.arange(len(sizes_present))
    width = 0.25
    for i, (col, label, color) in enumerate(zip(
            ['gurobi_time', 'ts_time', 'ga_time'],
            ['Gurobi', 'Tabu Search', 'Genetic Algorithm'],
            [GRB_COLOR, TS_COLOR, GA_COLOR])):
        means = [valid[valid['size'] == s][col].mean() for s in sizes_present]
        stds  = [valid[valid['size'] == s][col].std()  for s in sizes_present]
        bars = ax.bar(x + i * width, means, width,
                      label=label, color=color, alpha=0.8, edgecolor='white',
                      linewidth=1.2, yerr=stds, capsize=5, error_kw={'linewidth': 1.5})
    ax.set_xlabel('Instance Size', fontsize=12, fontweight='bold')
    ax.set_ylabel('Computation Time (seconds)', fontsize=12, fontweight='bold')
    ax.set_title('Mean Computation Time: Gurobi vs TS vs GA', fontsize=13, fontweight='bold')
    ax.set_xticks(x + width)
    ax.set_xticklabels([s.capitalize() for s in sizes_present], fontsize=11)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    fig.savefig(os.path.join(output_folder, 'computation_time.png'), dpi=150, bbox_inches='tight')
    plt.close()

    # ── 4. Speedup Factor ────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(14, 6))
    for i, (col, label, color) in enumerate(zip(
            ['ts_speedup_factor', 'ga_speedup_factor'],
            ['Tabu Search Speedup', 'GA Speedup'],
            [TS_COLOR, GA_COLOR])):
        means = [valid[valid['size'] == s][col].mean() for s in sizes_present]
        stds  = [valid[valid['size'] == s][col].std()  for s in sizes_present]
        bars = ax.bar(x + i * width, means, width,
                      label=label, color=color, alpha=0.8, edgecolor='white',
                      linewidth=1.2, yerr=stds, capsize=5, error_kw={'linewidth': 1.5})
    ax.axhline(y=1, color='gray', linestyle='--', alpha=0.6, label='Baseline (1×)')
    ax.set_xlabel('Instance Size', fontsize=12, fontweight='bold')
    ax.set_ylabel('Speedup over Gurobi (×)', fontsize=12, fontweight='bold')
    ax.set_title('Speedup Factor: TS and GA vs Gurobi', fontsize=13, fontweight='bold')
    ax.set_xticks(x + width / 2)
    ax.set_xticklabels([s.capitalize() for s in sizes_present], fontsize=11)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    fig.savefig(os.path.join(output_folder, 'speedup_factor.png'), dpi=150, bbox_inches='tight')
    plt.close()

    # ── 5. TS vs GA Head-to-Head Scatter ─────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 8))
    size_markers = {'small': 'o', 'medium': 's', 'large': '^'}
    diff_alphas  = {'easy': 0.4, 'moderate': 0.7, 'expensive': 1.0}
    for _, row in valid.iterrows():
        marker = size_markers.get(row['size'], 'o')
        alpha  = diff_alphas.get(row['difficulty'], 0.7)
        ax.scatter(row['ts_gap_percent'], row['ga_gap_percent'],
                   s=80, marker=marker, alpha=alpha,
                   c=GA_COLOR, edgecolors='black', linewidths=0.8)
    max_gap = max(valid[['ts_gap_percent', 'ga_gap_percent']].max())
    diag = np.linspace(0, max_gap * 1.05, 100)
    ax.plot(diag, diag, 'k--', alpha=0.5, label='TS = GA')
    ax.set_xlabel('Tabu Search Gap (%)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Genetic Algorithm Gap (%)', fontsize=12, fontweight='bold')
    ax.set_title('TS vs GA: Optimality Gap per Instance\n'
                 '(points above diagonal: TS outperforms GA)', fontsize=13, fontweight='bold')
    # Legend patches
    import matplotlib.patches as mpatches
    legend_elements = [
        mpatches.Patch(color='none', label='Size: o=Small  s=Medium  ^=Large'),
        mpatches.Patch(color='none', label='Difficulty: lighter=easier  darker=expensive'),
        plt.Line2D([0], [0], linestyle='--', color='black', alpha=0.5, label='TS = GA'),
    ]
    ax.legend(handles=legend_elements, fontsize=10, loc='upper left')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(os.path.join(output_folder, 'ts_vs_ga_scatter.png'), dpi=150, bbox_inches='tight')
    plt.close()

    # ── 6. Gap Heatmaps ──────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    for ax, col, label, cmap in zip(
            axes,
            ['ts_gap_percent', 'ga_gap_percent'],
            ['Tabu Search', 'Genetic Algorithm'],
            ['Blues', 'Oranges']):
        pivot = valid.pivot_table(values=col, index='size', columns='difficulty',
                                   aggfunc='mean')
        pivot = pivot.reindex(index=[s for s in SIZE_ORDER if s in pivot.index],
                              columns=[d for d in DIFF_ORDER if d in pivot.columns])
        sns.heatmap(pivot, ax=ax, annot=True, fmt='.2f', cmap=cmap,
                    linewidths=0.5, linecolor='white',
                    cbar_kws={'label': 'Mean Gap (%)'}, annot_kws={'size': 12})
        ax.set_title(f'{label}: Mean Optimality Gap (%) — Size × Difficulty',
                     fontsize=12, fontweight='bold')
        ax.set_xlabel('Difficulty', fontsize=11)
        ax.set_ylabel('Instance Size', fontsize=11)
    plt.tight_layout()
    fig.savefig(os.path.join(output_folder, 'gap_heatmap.png'), dpi=150, bbox_inches='tight')
    plt.close()

    # ── 7. Summary table ─────────────────────────────────────────────────────
    print("\n" + "=" * 75)
    print("VISUALIZATION SUMMARY — Mean Results by Instance Size")
    print("=" * 75)
    print(f"\n{'Size':<12} {'Gurobi (s)':>12} {'TS Gap%':>10} {'TS (s)':>10} "
          f"{'GA Gap%':>10} {'GA (s)':>10}")
    print("-" * 75)
    for s in sizes_present:
        sub = valid[valid['size'] == s]
        print(f"{s.capitalize():<12} {sub['gurobi_time'].mean():>12.2f} "
              f"{sub['ts_gap_percent'].mean():>10.2f} {sub['ts_time'].mean():>10.2f} "
              f"{sub['ga_gap_percent'].mean():>10.2f} {sub['ga_time'].mean():>10.2f}")

    print("\n" + "=" * 75)
    print(f"Plots saved to: {output_folder}/")
    print("=" * 75)


print("✅ Visualization module (3-way comparison: TS, GA, Gurobi) loaded successfully.")


---

## 10. Student Configuration

**Set your parameters in the cell below, then run it before running Section 11 (Main Execution).**

### 10.1  Gurobi Version

| Setting | When to use |
|---------|-------------|
| `"academic"` | You have an Academic or Full Gurobi license (no size limits). |
| `"free"`     | You are using the Gurobi free/trial version (≤ 2,000 variables). Large instances will be skipped. |

### 10.2  Tabu Search Hyperparameters

| Parameter | Default | Effect |
|-----------|---------|--------|
| `TS_MAX_ITERATIONS` | 200 | Total number of TS moves attempted |
| `TS_TABU_TENURE` | 10 | How many iterations a move stays on the tabu list |
| `TS_MAX_CANDIDATES` | 20 | Number of neighbor moves evaluated per iteration |
| `TS_MAX_NO_IMPROVE` | 40 | Stop after this many non-improving iterations |

### 10.3  Genetic Algorithm Hyperparameters

| Parameter | Default | Effect |
|-----------|---------|--------|
| `GA_POPULATION_SIZE` | 30 | Number of chromosomes per generation |
| `GA_MAX_GENERATIONS` | 100 | Maximum number of generations to evolve |
| `GA_CROSSOVER_RATE` | 0.85 | Probability that two parents produce offspring |
| `GA_MUTATION_RATE` | 0.10 | Per-gene mutation probability |
| `GA_MAX_NO_IMPROVE` | 30 | Early stopping if best cost does not improve |

### 10.4  Expected Runtimes (rough estimates)

| Instance size | Gurobi | Tabu Search | Genetic Algorithm |
|---------------|--------|-------------|-------------------|
| Small | < 5 s | < 3 s | < 5 s |
| Medium | 10–120 s | < 10 s | < 15 s |
| Large | 60–300 s | < 30 s | < 40 s |

> **Note**: These are approximate. Actual runtimes vary with instance difficulty and your machine's speed.


In [ ]:
# ════════════════════════════════════════════════════════════════
# STUDENT CONFIGURATION — Edit these values before running main()
# ════════════════════════════════════════════════════════════════

# ── Gurobi version ───────────────────────────────────────────────
GUROBI_VERSION = "academic"   # "academic" or "free"

# ── Instance generation ───────────────────────────────────────────
INSTANCES_FOLDER       = "cflp_instances"
INSTANCE_SIZES         = ['small', 'medium', 'large']
INSTANCE_DIFFICULTIES  = ['easy', 'moderate', 'expensive']
INSTANCES_PER_COMBO    = 16    # 3 × 3 × 16 = 144 instances total
RANDOM_SEED            = 42

# ── Tabu Search hyperparameters ───────────────────────────────────
TS_MAX_ITERATIONS  = 200   # Total TS iterations
TS_TABU_TENURE     = 10    # Tabu list tenure (iterations)
TS_MAX_CANDIDATES  = 20    # Candidate moves per iteration
TS_MAX_NO_IMPROVE  = 40    # Early stopping after no improvement

# ── Genetic Algorithm hyperparameters ─────────────────────────────
GA_POPULATION_SIZE = 30    # Number of chromosomes
GA_MAX_GENERATIONS = 100   # Maximum generations
GA_CROSSOVER_RATE  = 0.85  # Probability of crossover
GA_MUTATION_RATE   = 0.10  # Per-gene mutation probability
GA_MAX_NO_IMPROVE  = 30    # Early stopping after no improvement

# ── Gurobi time limit ─────────────────────────────────────────────
GUROBI_TIME_LIMIT  = 300   # seconds per instance

# ── Output settings ───────────────────────────────────────────────
RESULTS_FILE              = "results_comparison.csv"
SAVE_CONVERGENCE_PLOTS    = True
CONVERGENCE_PLOTS_FOLDER  = "convergence_plots"
VISUALIZATION_FOLDER      = "visualization_plots"

# ════════════════════════════════════════════════════════════════
print("✅ Configuration loaded:")
print(f"   Gurobi version : {GUROBI_VERSION}")
print(f"   Instances      : {INSTANCES_PER_COMBO} per combo "
      f"({len(INSTANCE_SIZES)}×{len(INSTANCE_DIFFICULTIES)}×{INSTANCES_PER_COMBO} = "
      f"{len(INSTANCE_SIZES)*len(INSTANCE_DIFFICULTIES)*INSTANCES_PER_COMBO} total)")
print(f"   TS params      : max_iter={TS_MAX_ITERATIONS}, tenure={TS_TABU_TENURE}, "
      f"max_cand={TS_MAX_CANDIDATES}, max_no_improve={TS_MAX_NO_IMPROVE}")
print(f"   GA params      : pop={GA_POPULATION_SIZE}, gen={GA_MAX_GENERATIONS}, "
      f"cx={GA_CROSSOVER_RATE}, mut={GA_MUTATION_RATE}, max_no_improve={GA_MAX_NO_IMPROVE}")
print(f"   Gurobi limit   : {GUROBI_TIME_LIMIT}s")


---

## 11. Main Execution

This cell orchestrates the full pipeline:
1. Generate all 144 CFLP instances (3 sizes × 3 difficulties × 16 each).
2. Solve each instance with **Gurobi**, **Tabu Search**, and **Genetic Algorithm**.
3. Save results to CSV + Excel.
4. Generate all visualization plots.

> **Before running**: Make sure you have run **Section 10 (Student Configuration)** first to set your Gurobi version and hyperparameters.


In [ ]:
def main():
    """
    Main execution pipeline: generate instances, run Gurobi + TS + GA, visualize results.
    """
    print("=" * 70)
    print("CFLP — Gurobi vs Tabu Search vs Genetic Algorithm")
    print("=" * 70)

    # ── Step 1: Generate instances ────────────────────────────────────────────
    print("\nStep 1: Generating CFLP instances...")
    generate_all_instances(
        output_folder=INSTANCES_FOLDER,
        sizes=INSTANCE_SIZES,
        difficulties=INSTANCE_DIFFICULTIES,
        instances_per_combo=INSTANCES_PER_COMBO,
        seed=RANDOM_SEED
    )

    # ── Step 2: Run experiments ───────────────────────────────────────────────
    print("\nStep 2: Running experiments (Gurobi + Tabu Search + Genetic Algorithm)...")
    results_df = run_experiments(
        instances_folder=INSTANCES_FOLDER,
        results_file=RESULTS_FILE,
        ts_max_iterations=TS_MAX_ITERATIONS,
        ts_tabu_tenure=TS_TABU_TENURE,
        ts_max_candidates=TS_MAX_CANDIDATES,
        ts_max_no_improve=TS_MAX_NO_IMPROVE,
        ga_population_size=GA_POPULATION_SIZE,
        ga_max_generations=GA_MAX_GENERATIONS,
        ga_crossover_rate=GA_CROSSOVER_RATE,
        ga_mutation_rate=GA_MUTATION_RATE,
        ga_max_no_improve=GA_MAX_NO_IMPROVE,
        gurobi_time_limit=GUROBI_TIME_LIMIT,
        save_convergence_plots=SAVE_CONVERGENCE_PLOTS,
        convergence_plots_folder=CONVERGENCE_PLOTS_FOLDER,
        gurobi_version=GUROBI_VERSION
    )

    # ── Step 3: Visualize results ─────────────────────────────────────────────
    print("\nStep 3: Generating visualization plots...")
    visualize_results(results_df, output_folder=VISUALIZATION_FOLDER)

    # ── Step 4: Print final summary ───────────────────────────────────────────
    valid = results_df[
        (results_df['gurobi_status'] == 'Optimal') &
        (results_df['ts_status'] == 'Completed') &
        (results_df['ga_status'] == 'Completed')
    ]

    print("\n" + "=" * 70)
    print("FINAL SUMMARY")
    print("=" * 70)
    print(f"Total instances: {len(results_df)}")
    print(f"Solved by all three methods: {len(valid)}")
    if len(valid) > 0:
        print(f"\n{'Metric':<35} {'Tabu Search':>15} {'Gen. Algorithm':>16}")
        print("-" * 70)
        print(f"{'Mean optimality gap (%):':<35} "
              f"{valid['ts_gap_percent'].mean():>14.2f}% "
              f"{valid['ga_gap_percent'].mean():>15.2f}%")
        print(f"{'Mean computation time (s):':<35} "
              f"{valid['ts_time'].mean():>15.2f} "
              f"{valid['ga_time'].mean():>16.2f}")
        print(f"{'Mean speedup over Gurobi (×):':<35} "
              f"{valid['ts_speedup_factor'].mean():>15.2f} "
              f"{valid['ga_speedup_factor'].mean():>16.2f}")
        print(f"\n{'Gurobi mean time (s):':<35} {valid['gurobi_time'].mean():>15.2f}")

    print("\n" + "=" * 70)
    print("Output files:")
    print(f"  Results CSV  : {RESULTS_FILE}")
    print(f"  Results XLSX : {RESULTS_FILE.replace('.csv', '.xlsx')}")
    print(f"  Convergence  : {CONVERGENCE_PLOTS_FOLDER}/")
    print(f"  Visualizations: {VISUALIZATION_FOLDER}/")
    print("=" * 70)
    print("\n✅ Done! Review the output files and answer the analysis questions.")

    return results_df


# Run the pipeline
results = main()


---

## 12. Bonus Challenges (Optional — Up to +10% Extra Credit)

These optional challenges require you to **systematically evaluate all proposed enhancements**, then **select and fully implement the single best one** for each algorithm. Simply picking one without comparing the others will not receive full marks.

---

### Bonus A — Improve Tabu Search (+5%)

**Step 1 — Explore all options.** Run a quick experiment (e.g., on the small/moderate subset only) for **each** of the four enhancements below. Record the mean gap improvement per option.

| Enhancement | Description |
|-------------|-------------|
| **Adaptive tenure** | Dynamically increase tenure when the search stagnates; decrease it when improvements are frequent. |
| **Aspiration+ criterion** | Allow a tabu move if it improves the **global** best by a margin δ (stronger than the standard criterion). |
| **Frequency-based memory** | Track long-term move frequencies and add a penalty to discourage frequently revisited configurations. |
| **Diversification restart** | After a stagnation threshold, apply a strong multi-swap perturbation to escape local optima. |

**Step 2 — Choose ONE.** Select the enhancement with the best gap reduction in Step 1 and justify your choice in the student guide (Section 12, Bonus A).

**Step 3 — Full implementation.** Apply your chosen enhancement to the full 144-instance experiment. Compare mean gap and runtime before vs after and report in Section 12 of the student guide.

---

### Bonus B — Improve Genetic Algorithm (+5%)

**Step 1 — Explore all options.** Run a quick experiment on the small/moderate subset for **each** of the four enhancements below. Record the mean gap improvement per option.

| Enhancement | Description |
|-------------|-------------|
| **Adaptive mutation** | Decrease mutation rate as population converges; increase it if diversity drops below a threshold. |
| **Two-point crossover** | Replace uniform crossover with two-point crossover; compare convergence speed and final gap. |
| **Memetic GA (TS + GA)** | Apply a short Tabu Search local refinement to the best chromosome each generation. |
| **Diversity preservation** | Track Hamming distance between chromosomes; actively reintroduce diversity when population converges. |

**Step 2 — Choose ONE.** Select the enhancement with the best gap reduction in Step 1 and justify your choice in the student guide (Section 12, Bonus B).

**Step 3 — Full implementation.** Apply your chosen enhancement to the full 144-instance experiment. Compare mean gap and runtime before vs after and report in Section 12 of the student guide.
